## Year, Month, day만 입력하면, 해당 날짜에 있던, 모든 경기의 game_idx 저장 -> unique_game_idx 리스트로 저장

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()


import requests
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import re
import json
import os
from datetime import date, timedelta
import pytz
import time
from datetime import datetime

BASE_URL = "https://www.korea-baseball.com/game/calendar"
KIND_CD = 31
SAVE_FILE = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/2026/game_idx_log.json"

# 수집 시작일 (최초 실행 시 기준점)
COLLECTION_START = date(2026, 3, 1)


def load_log():
    """저장된 로그 파일 불러오기"""
    if os.path.exists(SAVE_FILE):
        with open(SAVE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"last_collected_date": None, "game_ids": []}


def save_log(log):
    """로그 파일 저장"""
    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)


import socket


def ensure_dns(host):
    """Colab 런타임의 DNS가 이 호스트를 못 풀 때가 있어, 공용 DoH 로 A레코드를 직접 받아
    getaddrinfo 에 심어둔다(SNI·인증서 검증은 그대로 유지된다).

    반환값: 우회로 잡아둔 IP. 시스템 리졸버가 멀쩡하거나 DoH 도 실패하면 None.
    """
    overrides = getattr(socket, "_dns_override", None)
    if overrides is None:
        overrides = socket._dns_override = {}
        _orig_getaddrinfo = socket.getaddrinfo

        def _getaddrinfo(h, port, *args, **kwargs):
            if h in socket._dns_override:
                return [(socket.AF_INET, socket.SOCK_STREAM, 6, "",
                         (socket._dns_override[h], port))]
            return _orig_getaddrinfo(h, port, *args, **kwargs)

        socket.getaddrinfo = _getaddrinfo

    if host in overrides:
        return overrides[host]

    try:
        socket.getaddrinfo(host, 443, socket.AF_INET, socket.SOCK_STREAM)
        return None
    except socket.gaierror:
        pass

    ip = None
    for resolver in ("https://8.8.8.8/resolve", "https://1.1.1.1/dns-query"):
        try:
            r = requests.get(resolver,
                             params={"name": host, "type": "A"},
                             headers={"accept": "application/dns-json"},
                             timeout=10)
            ip = next((a["data"] for a in r.json().get("Answer", [])
                       if a.get("type") == 1), None)
        except Exception:
            ip = None
        if ip:
            break
    if not ip:
        return None

    overrides[host] = ip
    return ip


def get_game_ids_for_month(year, month, target_days, existing_ids):
    """특정 연월의 특정 날짜들에서 game_idx 수집 (중복 제외)"""
    url = f"{BASE_URL}?month={month}&year={year}&kind_cd={KIND_CD}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    # Colab 런타임이 이 호스트를 못 푸는 일이 있어, DNS 우회를 걸어가며 재시도한다
    res = None
    max_retries = 5
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers=headers, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt >= max_retries - 1:
                raise e
            ip = ensure_dns(urlparse(BASE_URL).hostname)
            wait = 2 if ip else 5 * (attempt + 1)
            detail = f", DoH로 {ip} 확보" if ip else ""
            print(f"    캘린더 연결 실패 ({attempt+1}/{max_retries}){detail}, {wait}초 후 재시도...")
            time.sleep(wait)
    soup = BeautifulSoup(res.text, "html.parser")
    date_sch = soup.select_one("div.date_sch ol.sch")
    if not date_sch:
        return []

    new_ids = []
    for day in target_days:
        target_date = f"{year}:{month:02d}:{day:02d}"
        for li in date_sch.select("li"):
            if li.get("date", "").startswith(target_date):
                a_tag = li.select_one("a[href]")
                if a_tag:
                    match = re.search(r"game_idx=(\d+)", a_tag["href"])
                    if match:
                        game_id = int(match.group(1))
                        if game_id not in existing_ids and game_id not in new_ids:
                            new_ids.append(game_id)
    return new_ids


def collect_until_yesterday():
    log = load_log()
    existing_ids = log["game_ids"]

    # 시작일 결정
    if log["last_collected_date"]:
        start_date = date.fromisoformat(log["last_collected_date"]) + timedelta(days=1)
    else:
        start_date = COLLECTION_START

    kst = pytz.timezone('Asia/Seoul')
    end_date = datetime.now(kst).date()

    if start_date > end_date:
        print(f"이미 최신 상태입니다. (마지막 수집일: {log['last_collected_date']})")
        print(f"총 {len(existing_ids)}경기 저장됨: {existing_ids}")
        return existing_ids

    print(f"수집 범위: {start_date} ~ {end_date}")

    # 연월별로 묶어서 요청
    from itertools import groupby
    from calendar import monthrange

    all_dates = []
    cur = start_date
    while cur <= end_date:
        all_dates.append(cur)
        cur += timedelta(days=1)

    # 연-월별 그룹핑
    from collections import defaultdict
    month_groups = defaultdict(list)
    for d in all_dates:
        month_groups[(d.year, d.month)].append(d.day)

    new_ids_total = []
    for (year, month), days in sorted(month_groups.items()):
        print(f"  {year}년 {month}월 {days} 수집 중...")
        new_ids = get_game_ids_for_month(year, month, days, existing_ids + new_ids_total)
        new_ids_total.extend(new_ids)
        print(f"    → {len(new_ids)}경기 추가")

    # 로그 업데이트
    log["game_ids"] = existing_ids + new_ids_total
    log["last_collected_date"] = end_date.isoformat()
    save_log(log)

    print(f"\n신규 수집: {len(new_ids_total)}경기")
    print(f"누적 총계: {len(log['game_ids'])}경기")
    print(f"전체 game_ids: {log['game_ids']}")
    return log["game_ids"]


collect_until_yesterday()

In [ ]:
import requests
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import re
import json
import os
import pandas as pd
import time
import random
import pytz
import unicodedata
from datetime import date, timedelta, datetime
from collections import defaultdict


# ==================== 설정 ====================
BASE_URL = "https://www.korea-baseball.com/game/calendar"
KIND_CD = 31
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/2026"
LOG_FILE = os.path.join(BASE_DIR, "collection_log.json")
COLLECTION_START = date(2026, 3, 1)
GAMES_PER_FILE = 10
ALPHABET = "abcdefghijklmnopqrstuvwxyz"
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')
LEAGUE_MISSING_LOG_FILE = os.path.join(BASE_DIR, "league_missing_game_ids.txt")

EXCLUDE_DATES = []


# ==================== KST 날짜 ====================
def get_kst_today():
    kst = pytz.timezone('Asia/Seoul')
    return datetime.now(kst).date()


def _to_date(d):
    """문자열('YYYY-MM-DD') 또는 date 객체를 date로 통일"""
    if d is None:
        return None
    if isinstance(d, str):
        return date.fromisoformat(d)
    return d


# ==================== 로그 관리 ====================
def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            log = json.load(f)
        if "suspended_game_ids" not in log:
            log["suspended_game_ids"] = []
        return log
    return {
        "last_collected_date": None,
        "game_ids": [],
        "failed_game_ids": [],
        "suspended_game_ids": [],
        "month_file_status": {}
    }


def save_log(log):
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)


# ==================== game_idx 수집 ====================
import socket


def ensure_dns(host):
    """Colab 런타임의 DNS가 이 호스트를 못 풀 때가 있어, 공용 DoH 로 A레코드를 직접 받아
    getaddrinfo 에 심어둔다(SNI·인증서 검증은 그대로 유지된다).

    반환값: 우회로 잡아둔 IP. 시스템 리졸버가 멀쩡하거나 DoH 도 실패하면 None.
    """
    overrides = getattr(socket, "_dns_override", None)
    if overrides is None:
        overrides = socket._dns_override = {}
        _orig_getaddrinfo = socket.getaddrinfo

        def _getaddrinfo(h, port, *args, **kwargs):
            if h in socket._dns_override:
                return [(socket.AF_INET, socket.SOCK_STREAM, 6, "",
                         (socket._dns_override[h], port))]
            return _orig_getaddrinfo(h, port, *args, **kwargs)

        socket.getaddrinfo = _getaddrinfo

    if host in overrides:
        return overrides[host]

    try:
        socket.getaddrinfo(host, 443, socket.AF_INET, socket.SOCK_STREAM)
        return None
    except socket.gaierror:
        pass

    ip = None
    for resolver in ("https://8.8.8.8/resolve", "https://1.1.1.1/dns-query"):
        try:
            r = requests.get(resolver,
                             params={"name": host, "type": "A"},
                             headers={"accept": "application/dns-json"},
                             timeout=10)
            ip = next((a["data"] for a in r.json().get("Answer", [])
                       if a.get("type") == 1), None)
        except Exception:
            ip = None
        if ip:
            break
    if not ip:
        return None

    overrides[host] = ip
    return ip


def get_game_ids_for_month(year, month, target_days, existing_ids):
    url = f"{BASE_URL}?month={month}&year={year}&kind_cd={KIND_CD}"
    headers = {"User-Agent": "Mozilla/5.0"}

    # Colab 런타임이 이 호스트를 못 푸는 일이 있어, DNS 우회를 걸어가며 재시도한다
    res = None
    max_retries = 5
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers=headers, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt >= max_retries - 1:
                raise e
            ip = ensure_dns(urlparse(BASE_URL).hostname)
            wait = 2 if ip else 5 * (attempt + 1)
            detail = f", DoH로 {ip} 확보" if ip else ""
            print(f"    캘린더 연결 실패 ({attempt+1}/{max_retries}){detail}, {wait}초 후 재시도...")
            time.sleep(wait)
    soup = BeautifulSoup(res.text, "html.parser")
    date_sch = soup.select_one("div.date_sch ol.sch")
    if not date_sch:
        return []

    new_ids = []
    for day in target_days:
        target_date = f"{year}:{month:02d}:{day:02d}"
        for li in date_sch.select("li"):
            if li.get("date", "").startswith(target_date):
                a_tag = li.select_one("a[href]")
                if a_tag:
                    match = re.search(r"game_idx=(\d+)", a_tag["href"])
                    if match:
                        game_id = int(match.group(1))
                        if game_id not in existing_ids and game_id not in new_ids:
                            new_ids.append(game_id)
    return new_ids


def get_all_saved_game_ids():
    """
    BASE_DIR 하위의 모든 PBP 엑셀 파일(YYYY_MM_x.xlsx)을 직접 스캔해서
    '실제로 저장된' game_id 집합을 반환한다.

    log["game_ids"]는 달력에서 '발견된' 경기 목록일 뿐이라, 크롤링 실패나
    일시정지로 엑셀에는 한 줄도 안 들어간 game_id도 섞여 있을 수 있다.
    재수집 여부 판단은 반드시 이 함수(실제 파일 기준)로 해야 한다.
    """
    saved_ids = set()
    if not os.path.exists(BASE_DIR):
        return saved_ids

    for root, dirs, files in os.walk(BASE_DIR):
        for fname in files:
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue
            fpath = os.path.join(root, fname)
            try:
                df = pd.read_excel(fpath, engine="openpyxl", usecols=["game_id"])
                if df.empty:
                    continue
                ids = pd.to_numeric(df["game_id"], errors="coerce").dropna().astype(int).tolist()
                saved_ids.update(ids)
            except Exception as e:
                print(f"  [경고] {fname_nfc} 읽기 실패, 스킵: {e}")
    return saved_ids


def _collect_game_ids_for_dates(log, all_dates):
    """공통 로직: 주어진 날짜 리스트(all_dates)에서 신규 game_idx를 수집해 log에 반영

    log["game_ids"]에 없더라도 '이미 엑셀에 저장돼 있는' game_id는 재수집 대상에서 뺀다.
    retry_failed_games()처럼 달력을 훑지 않고 game_id를 직접 넣어 저장한 경기는
    log["game_ids"]에 들어가지 않기 때문에, 이 체크가 없으면 나중에 run_pipeline()이
    그 날짜를 훑을 때 '신규'로 다시 잡아 같은 경기를 엑셀에 중복으로 append하게 된다.
    (save_games_to_files()는 기존 파일과의 game_id 중복 검사를 하지 않는다.)
    """
    existing_ids = log["game_ids"]
    saved_ids = get_all_saved_game_ids()

    month_groups = defaultdict(list)
    for d in all_dates:
        month_groups[(d.year, d.month)].append(d.day)

    new_ids = []
    already_saved_ids = []
    for (year, month), days in sorted(month_groups.items()):
        print(f"  {year}년 {month}월 수집 중... ({len(days)}일)")
        ids = get_game_ids_for_month(
            year, month, days, existing_ids + new_ids + already_saved_ids
        )
        fresh = [gid for gid in ids if gid not in saved_ids]
        skipped = [gid for gid in ids if gid in saved_ids]
        new_ids.extend(fresh)
        already_saved_ids.extend(skipped)
        print(f"    → {len(fresh)}경기 추가")
        if skipped:
            print(f"    (이미 엑셀에 저장돼 있어 스킵: {len(skipped)}경기) {skipped}")

    # 스킵한 경기도 '발견된 목록'에는 누적해 둔다 (다음부터는 엑셀 스캔 없이도 걸러짐)
    log["game_ids"] = existing_ids + new_ids + already_saved_ids
    save_log(log)
    return new_ids


def collect_new_game_ids(log):
    """
    [일반 모드] last_collected_date + 1 ~ 오늘까지 신규 game_idx 수집.
    last_collected_date는 여기서 갱신하지 않고 end_date만 반환.
    PBP 저장 완료 후 run_pipeline()에서 갱신함.
    """
    if log["last_collected_date"]:
        start_date = date.fromisoformat(log["last_collected_date"]) + timedelta(days=1)
    else:
        start_date = COLLECTION_START

    end_date = get_kst_today()  # 당일 포함

    if start_date > end_date:
        print(f"이미 최신 상태입니다. (마지막 수집일: {log['last_collected_date']})")
        return [], None

    print(f"game_idx 수집 범위: {start_date} ~ {end_date}")
    if EXCLUDE_DATES:
        print(f"제외 날짜: {[str(d) for d in EXCLUDE_DATES]}")

    all_dates = []
    cur = start_date
    while cur <= end_date:
        if cur not in EXCLUDE_DATES:
            all_dates.append(cur)
        cur += timedelta(days=1)

    new_ids = _collect_game_ids_for_dates(log, all_dates)
    print(f"신규 game_idx {len(new_ids)}개 수집 완료\n")
    return new_ids, end_date


def collect_game_ids_for_range(log, start_date, end_date):
    """
    [수동 범위 모드] 사용자가 지정한 start_date ~ end_date 구간을 다시 수집.

    핵심: 이 구간에 해당하는 game_idx는 log["game_ids"]에 이미 있더라도
    무조건 다시 후보로 잡는다. 대신 '실제로 엑셀 파일에 저장되어 있는지'
    (get_all_saved_game_ids() 기준)만 확인해서, 이미 엑셀에 들어간 경기만
    스킵하고 나머지(크롤링 실패/일시정지/누락 등으로 엑셀에 없는 경기)는
    전부 다시 크롤링 대상으로 포함시킨다.

    즉, 해당 날짜 범위는 "이 날짜를 이미 훑어봤다"는 이유로 스킵되지 않고,
    "이 game_idx가 실제로 파일에 저장돼 있다"는 것만 스킵 기준이 된다.

    last_collected_date는 여기서 갱신하지 않음 (run_pipeline에서 전진(forward)만 허용해서 갱신).
    """
    start_date = _to_date(start_date)
    end_date = _to_date(end_date)

    if start_date > end_date:
        print(f"잘못된 날짜 범위입니다: {start_date} ~ {end_date}")
        return [], None

    print(f"[수동 범위 재수집] 날짜 범위: {start_date} ~ {end_date}")
    if EXCLUDE_DATES:
        print(f"제외 날짜: {[str(d) for d in EXCLUDE_DATES]}")

    all_dates = []
    cur = start_date
    while cur <= end_date:
        if cur not in EXCLUDE_DATES:
            all_dates.append(cur)
        cur += timedelta(days=1)

    month_groups = defaultdict(list)
    for d in all_dates:
        month_groups[(d.year, d.month)].append(d.day)

    # 1) 달력에서 이 구간의 game_idx를 전부 가져온다 (existing_ids 체크 없이 전부 조회)
    found_ids = []
    for (year, month), days in sorted(month_groups.items()):
        print(f"  {year}년 {month}월 조회 중... ({len(days)}일)")
        ids = get_game_ids_for_month(year, month, days, existing_ids=[])
        found_ids.extend(ids)
        print(f"    → {len(ids)}경기 발견 (날짜 기준, 중복 제거 전)")

    found_ids = list(dict.fromkeys(found_ids))  # 순서 유지하며 중복 제거

    # 2) 실제로 엑셀에 저장돼 있는 game_id만 제외하고, 나머지는 전부 재수집 대상
    print("  엑셀 파일 스캔하여 실제 저장 여부 확인 중...")
    saved_ids = get_all_saved_game_ids()
    new_ids = [gid for gid in found_ids if gid not in saved_ids]
    skipped = len(found_ids) - len(new_ids)
    print(f"  이미 엑셀에 저장됨: {skipped}경기 스킵 / 재수집 대상: {len(new_ids)}경기")

    # log["game_ids"]는 참고용 발견 목록으로 누적만 해둔다 (dedup 기준으로는 사용하지 않음)
    existing_logged = set(log["game_ids"])
    log["game_ids"] = list(existing_logged | set(found_ids))
    save_log(log)

    print(f"신규(재수집 대상) game_idx {len(new_ids)}개 (수동 범위)\n")
    return new_ids, end_date


# ==================== 예외_아웃 열 일괄 백필 ====================
def backfill_예외_아웃_column():
    """
    BASE_DIR 하위 모든 월별 폴더(예: 3월/, 4월/...)의 PBP 엑셀 파일(YYYY_MM_x.xlsx)을
    전부 스캔해서 order와 batter 사이에 예외_아웃 열이 있도록 맞춘다.
      - 이미 '예외_아웃' 열이 있으면 스킵
      - 과거에 'R예외_아웃'이라는 이름으로 추가된 열이 있으면 '예외_아웃'으로 이름만 변경
      - 둘 다 없으면 order 바로 다음에 빈 칸으로 새로 추가
    안전하게 여러 번 실행 가능.
    """
    if not os.path.exists(BASE_DIR):
        print(f"BASE_DIR이 존재하지 않습니다: {BASE_DIR}")
        return

    added_files = []
    renamed_files = []
    skipped_files = []
    error_files = []

    for root, dirs, files in os.walk(BASE_DIR):
        for fname in sorted(files):
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue

            fpath = os.path.join(root, fname)
            try:
                df = pd.read_excel(fpath, engine="openpyxl")

                if '예외_아웃' in df.columns:
                    skipped_files.append(fname_nfc)
                    continue

                if 'R예외_아웃' in df.columns:
                    df = df.rename(columns={'R예외_아웃': '예외_아웃'})
                    df.to_excel(fpath, index=False, engine="openpyxl")
                    renamed_files.append(fname_nfc)
                    print(f"  [이름변경] {fname_nfc}: R예외_아웃 → 예외_아웃")
                    continue

                if 'order' not in df.columns or 'batter' not in df.columns:
                    print(f"  [경고] {fname_nfc}: order/batter 컬럼 없음, 스킵")
                    error_files.append(fname_nfc)
                    continue

                order_idx = df.columns.get_loc('order')
                df.insert(order_idx + 1, '예외_아웃', '')

                df.to_excel(fpath, index=False, engine="openpyxl")
                added_files.append(fname_nfc)
                print(f"  [추가] {fname_nfc}")

            except Exception as e:
                print(f"  [오류] {fname_nfc} 처리 실패: {e}")
                error_files.append(fname_nfc)

    print("\n" + "=" * 60)
    print(f"예외_아웃 열 백필 완료")
    print(f"신규 추가: {len(added_files)} / 이름변경: {len(renamed_files)} / 이미 존재(스킵): {len(skipped_files)} / 오류: {len(error_files)}")
    if error_files:
        print(f"오류 파일: {error_files}")
    print("=" * 60)


# ==================== RBI 열 일괄 백필 ====================
def backfill_RBI_column():
    """
    BASE_DIR 하위 모든 월별 폴더(예: 3월/, 4월/...)의 PBP 엑셀 파일(YYYY_MM_x.xlsx)을
    전부 스캔해서 home_score와 position 사이에 RBI 열이 있도록 맞춘다.
      - 이미 'RBI' 열이 있으면 스킵
      - 없으면 home_score 바로 다음에 빈 칸으로 새로 추가
    안전하게 여러 번 실행 가능.
    """
    if not os.path.exists(BASE_DIR):
        print(f"BASE_DIR이 존재하지 않습니다: {BASE_DIR}")
        return

    added_files = []
    skipped_files = []
    error_files = []

    for root, dirs, files in os.walk(BASE_DIR):
        for fname in sorted(files):
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue

            fpath = os.path.join(root, fname)
            try:
                df = pd.read_excel(fpath, engine="openpyxl")

                if 'RBI' in df.columns:
                    skipped_files.append(fname_nfc)
                    continue

                if 'home_score' not in df.columns or 'position' not in df.columns:
                    print(f"  [경고] {fname_nfc}: home_score/position 컬럼 없음, 스킵")
                    error_files.append(fname_nfc)
                    continue

                home_score_idx = df.columns.get_loc('home_score')
                df.insert(home_score_idx + 1, 'RBI', '')

                df.to_excel(fpath, index=False, engine="openpyxl")
                added_files.append(fname_nfc)
                print(f"  [추가] {fname_nfc}")

            except Exception as e:
                print(f"  [오류] {fname_nfc} 처리 실패: {e}")
                error_files.append(fname_nfc)

    print("\n" + "=" * 60)
    print(f"RBI 열 백필 완료")
    print(f"신규 추가: {len(added_files)} / 이미 존재(스킵): {len(skipped_files)} / 오류: {len(error_files)}")
    if error_files:
        print(f"오류 파일: {error_files}")
    print("=" * 60)


# ==================== box_score 의 game_name(라운드/구장) ====================
def fetch_game_name_dd(game_idx, max_retries=3, retry_wait=30):
    """box_score 의 dl.game_name > dd 에서 (라운드, 구장)을 뽑는다. 실패하면 (None, None).

    dd 는 '2026.06.06 09:30 / 신월 야구공원 / 리그전' 처럼 슬래시로 세 칸이다
    (2020·2023·2025·2026, 주말리그·전국대회·준결승 표본 10경기 모두 동일한 형식).

    구장을 record_detail 대신 여기서 받는 이유: record_detail 의 dd 는 공백뿐이라
    '날짜 시각 구장' 을 공백으로 잘라야 하는데, 그러면 구장 이름의 생김새에 기대게 된다.
    실제로 '곤지암 팀업캠퍼스 제2구장' 이 '곤지암' 으로 잘려 저장된 시즌이 있었다.
    슬래시로 갈린 쪽은 이름이 아무리 길고 괄호가 섞여도 칸이 흔들리지 않는다.

    칸 수가 3이 아니면 형식이 바뀐 것이므로 아무것도 돌려주지 않고 경고만 찍는다 —
    엉뚱한 칸을 구장으로 저장해 조용히 오염시키는 것보다, 호출한 쪽이 record_detail
    값으로 폴백하는 편이 안전하다(구장명 오류는 몇 년이 지나도 소리를 내지 않는다).
    """
    url = f"https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"

    res = None
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt < max_retries - 1:
                print(f"  [{game_idx}] box_score 연결 실패 ({attempt+1}/{max_retries}), {retry_wait}초 후 재시도...")
                time.sleep(retry_wait)
            else:
                print(f"  [{game_idx}] box_score 연결 실패, 재시도 초과: {e}")
                return None, None
    if res is None:
        return None, None

    res.encoding = "utf-8"
    game_name_elem = BeautifulSoup(res.text, "html.parser").find("dl", class_="game_name")
    dd = game_name_elem.find("dd") if game_name_elem else None
    if not dd:
        return None, None

    text = dd.get_text(strip=True)
    parts = [p.strip() for p in text.split("/")]
    if len(parts) != 3:
        print(f"  ⚠ [{game_idx}] box_score dd 형식이 예상과 다릅니다(칸 {len(parts)}개): {text!r}")
        return None, None
    _, stadium, league = parts
    return (league or None), (stadium or None)


def fetch_league(game_idx, max_retries=3, retry_wait=30):
    """라운드(리그전/16강전/결승전)만 필요한 호출용 — backfill_league_column 이 쓴다."""
    league, _ = fetch_game_name_dd(game_idx, max_retries, retry_wait)
    return league


def backfill_league_column(base_dir=BASE_DIR, target_game_ids=None):
    """
    target_game_ids:
      - None(기본값): base_dir 하위 전체 game_id 대상
      - 단일 game_id (str 또는 int): 그 경기 하나만 대상
      - game_id 리스트: 나열된 경기들만 대상
    나머지 game_id의 기존 league/details 값은 건드리지 않는다.

    이미 league 값이 채워져 있는 game_id는 재요청 없이 건너뛴다.
    (중간에 멈췄다가 다시 실행해도 이어서 처리됨)
    """
    if target_game_ids is not None:
        if isinstance(target_game_ids, (str, int)):
            target_game_ids = [target_game_ids]
        target_game_ids = set(str(g) for g in target_game_ids)

    league_cache = {}
    missing_ids = []
    elapsed_log = {}

    for root, dirs, files in os.walk(base_dir):
        for fname in sorted(files):
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue

            fpath = os.path.join(root, fname)
            df = pd.read_excel(fpath, engine="openpyxl", dtype={"game_id": str})

            file_game_ids = df['game_id'].dropna().unique().tolist()
            if target_game_ids is not None:
                file_game_ids = [gid for gid in file_game_ids if gid in target_game_ids]
                if not file_game_ids:
                    continue

            if 'league' in df.columns:
                existing_league = (
                    df.dropna(subset=['game_id'])
                    .drop_duplicates(subset=['game_id'])
                    .set_index('game_id')['league']
                    .to_dict()
                )
            else:
                existing_league = {}

            for gid in file_game_ids:
                if gid in league_cache:
                    continue

                prior = existing_league.get(gid)
                if isinstance(prior, str) and prior.strip():
                    league_cache[gid] = prior
                    print(f"  [{gid}] league = {prior} (이미 완료됨, 스킵)")
                    continue

                start = time.time()
                league = fetch_league(gid)
                elapsed = time.time() - start
                elapsed_log[gid] = elapsed

                league_cache[gid] = league if league else ''
                if league:
                    print(f"  [{gid}] league = {league} ({elapsed:.2f}초)")
                else:
                    missing_ids.append(gid)
                    print(f"  [{gid}] league 정보를 찾지 못함 ({elapsed:.2f}초)")
                time.sleep(random.uniform(9, 11))

            if 'league' not in df.columns:
                df['league'] = ''

            mask = df['game_id'].isin(file_game_ids)
            df.loc[mask, 'league'] = df.loc[mask, 'game_id'].map(league_cache)
            df.loc[mask, 'details'] = (
                df.loc[mask, 'date'] + ' ' + df.loc[mask, 'time'] + ' ' +
                df.loc[mask, 'stadium'] + ' ' + df.loc[mask, 'league']
            )

            cols = list(df.columns)
            if cols.index('league') != cols.index('stadium') + 1:
                cols.remove('league')
                cols.insert(cols.index('stadium') + 1, 'league')
                df = df[cols]

            df.to_excel(fpath, index=False, engine="openpyxl")
            print(f"[완료] {fname_nfc}")

    if missing_ids:
        with open(LEAGUE_MISSING_LOG_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(missing_ids))
        print(f"\nleague 정보를 찾지 못한 game_id {len(missing_ids)}개 -> {LEAGUE_MISSING_LOG_FILE}")

    if elapsed_log:
        avg = sum(elapsed_log.values()) / len(elapsed_log)
        print(f"\n신규 요청 {len(elapsed_log)}건, 평균 {avg:.2f}초 / 최대 {max(elapsed_log.values()):.2f}초 / 최소 {min(elapsed_log.values()):.2f}초")


# ==================== 파일 경로 관리 ====================
def get_month_key(date_str):
    parts = date_str.split(".")
    return f"{parts[0]}_{parts[1]}"


def get_month_folder(month_key):
    month_num = int(month_key.split("_")[1])
    folder = os.path.join(BASE_DIR, f"{month_num}월")
    os.makedirs(folder, exist_ok=True)
    return folder


def get_file_path(month_key, file_letter):
    folder = get_month_folder(month_key)
    return os.path.join(folder, f"{month_key}_{file_letter}.xlsx")


def next_letter(letter):
    idx = ALPHABET.index(letter)
    return ALPHABET[idx + 1]


def load_existing_file(file_path):
    if os.path.exists(file_path):
        return pd.read_excel(file_path, engine="openpyxl", dtype={"game_id": str})
    return pd.DataFrame()


def get_last_file_in_month(month_key):
    """폴더에서 실제로 존재하는 가장 마지막 파일 경로 반환"""
    folder = get_month_folder(month_key)
    existing = sorted([
        f for f in os.listdir(folder)
        if re.match(rf'^{month_key}_[a-z]+\.xlsx$', f)
    ])
    if not existing:
        return None
    return os.path.join(folder, existing[-1])

# ============ PBP 파일 저장 ==================
def save_games_to_files(new_game_data_list, log):
    """
    월별로 그룹핑한 뒤, 그 달의 '마지막으로 존재하는 파일'을 실제로 읽어서
    game_id.nunique()로 빈 자리를 확인하고 맨 끝에 append.
    10경기가 차면 다음 알파벳 파일을 새로 만들어 이어서 채움.
    (날짜 순서와 무관하게 항상 끝에 붙이므로, 과거 날짜를 나중에 수집해도
    이미 존재하는 마지막 파일 뒤에 자연스럽게 이어붙는다.)
    """
    month_groups = defaultdict(list)
    for game_id, df in new_game_data_list:
        date_str = df["date"].iloc[0]
        month_key = get_month_key(date_str)
        month_groups[month_key].append((game_id, df))

    month_file_status = log.get("month_file_status", {})

    for month_key in sorted(month_groups.keys()):
        games = month_groups[month_key]
        month_num = int(month_key.split("_")[1])
        print(f"\n[{month_num}월] {len(games)}경기 저장 처리 중...")

        # 실제 폴더에 존재하는 마지막 파일을 우선 신뢰 (로그가 더 오래됐을 수 있으므로)
        last_existing = get_last_file_in_month(month_key)
        if last_existing is not None:
            current_letter = os.path.splitext(os.path.basename(last_existing))[0].split("_")[-1]
        elif month_key in month_file_status:
            current_letter = month_file_status[month_key]["last_file"]
        else:
            current_letter = "a"

        current_file_path = get_file_path(month_key, current_letter)

        if os.path.exists(current_file_path):
            pending_df = load_existing_file(current_file_path)
            current_count = pending_df["game_id"].nunique() if not pending_df.empty else 0
        else:
            pending_df = pd.DataFrame()
            current_count = 0

        # 혹시 이미 꽉 찬 파일을 읽어왔다면 즉시 다음 알파벳으로 넘어감
        while current_count >= GAMES_PER_FILE:
            current_letter = next_letter(current_letter)
            current_file_path = get_file_path(month_key, current_letter)
            if os.path.exists(current_file_path):
                pending_df = load_existing_file(current_file_path)
                current_count = pending_df["game_id"].nunique() if not pending_df.empty else 0
            else:
                pending_df = pd.DataFrame()
                current_count = 0

        for game_id, game_df in games:
            pending_df = pd.concat([pending_df, game_df], ignore_index=True)
            current_count += 1

            if current_count >= GAMES_PER_FILE:
                pending_df.to_excel(current_file_path, index=False, engine="openpyxl")
                print(f"  저장 완료: {os.path.basename(current_file_path)} ({current_count}경기)")

                current_letter = next_letter(current_letter)
                current_file_path = get_file_path(month_key, current_letter)
                pending_df = pd.DataFrame()
                current_count = 0

        if not pending_df.empty:
            pending_df.to_excel(current_file_path, index=False, engine="openpyxl")
            print(f"  저장 완료: {os.path.basename(current_file_path)} ({current_count}경기, 미완료)")

        month_file_status[month_key] = {
            "last_file": current_letter,
            "game_count": current_count
        }

    log["month_file_status"] = month_file_status
    save_log(log)

# ==================== 경기 크롤링 공통 함수 ====================
def crawl_games(game_ids):
    success_list = []
    fail_list = []
    suspended_list = []

    for i, game_id in enumerate(game_ids, 1):
        print(f"\n[{i}/{len(game_ids)}] game_idx={game_id}")
        result = extract_game_data(game_id)

        if isinstance(result, str) and result == "SUSPENDED":
            suspended_list.append(game_id)
            print(f"  일시정지 경기로 분류: game_idx={game_id} → suspended_game_ids에 저장")
        elif isinstance(result, pd.DataFrame) and not result.empty:
            success_list.append((game_id, result))
        else:
            fail_list.append(game_id)
            print(f"  실패: game_idx={game_id} 스킵")

        if i < len(game_ids):
            delay = random.uniform(10, 12)
            print(f"  {delay:.1f}초 대기...")
            time.sleep(delay)

    return success_list, fail_list, suspended_list

print("✓ crawl_games 재정의 완료")

# ==================== 경기 데이터 추출 ====================
def is_pure_substitute(series):
    return (
        series.str.contains('대주자', na=False) |
        (series.str.contains('대수비', na=False) & ~series.str.contains(',', na=False))
    )


def is_승부주자(series):
    return series.str.contains('승부주자', na=False)


def is_game_suspended(soup):
    suspend_btn = soup.find('a', class_='btn_rboxbk_big165_gray')
    if suspend_btn and '일시정지경기' in suspend_btn.get_text(strip=True):
        return True
    return False


def extract_pitcher_info(soup, tables):
    if len(tables) >= 8:
        투수_테이블_인덱스 = [3, 7]
    elif len(tables) >= 4:
        투수_테이블_인덱스 = [1, 3]
    else:
        return [], []

    pitchers_list = []
    for table_num in 투수_테이블_인덱스:
        if table_num >= len(tables):
            pitchers_list.append([])
            continue

        table = tables[table_num]
        tbody = table.find('tbody')
        if not tbody:
            pitchers_list.append([])
            continue

        team_pitchers = []
        for tr in tbody.find_all('tr'):
            if 'sum' in tr.get('class', []):
                continue
            cells = []
            for th in tr.find_all('th'):
                cells.append(th.get_text(strip=True))
            for td in tr.find_all('td'):
                cells.append(td.get_text(strip=True))
            if len(cells) < 7:
                continue
            투수명 = cells[0]
            타자수_str = cells[6]
            try:
                타자수 = int(타자수_str)
                team_pitchers.extend([투수명] * 타자수)
            except ValueError:
                continue
        pitchers_list.append(team_pitchers)

    team1_pitchers = pitchers_list[0] if len(pitchers_list) > 0 else []
    team2_pitchers = pitchers_list[1] if len(pitchers_list) > 1 else []
    return team1_pitchers, team2_pitchers


def apply_batting_order_continuity(df):
    result_rows = []
    for team_name in df['team'].unique():
        team_data = df[df['team'] == team_name].copy()
        team_data['is_substitute'] = is_pure_substitute(team_data['기록'])
        regular_pas = team_data[~team_data['is_substitute']].copy()
        substitute_pas = team_data[team_data['is_substitute']].copy()

        last_order = 0
        processed_regular = []
        for inning in sorted(regular_pas['이닝'].unique()):
            inning_data = regular_pas[regular_pas['이닝'] == inning].copy()
            plate_appearances = len(inning_data)
            inning_orders = []
            for i in range(plate_appearances):
                current_order = (last_order % 9) + 1
                inning_orders.append(current_order)
                last_order = current_order

            inning_data_sorted = inning_data.sort_values('타순').reset_index(drop=True)
            for idx, expected_order in enumerate(inning_orders):
                matching_rows = inning_data_sorted[inning_data_sorted['타순'] == expected_order]
                if len(matching_rows) > 0:
                    row_to_add = matching_rows.iloc[0].copy()
                    inning_data_sorted = inning_data_sorted.drop(matching_rows.index[0])
                else:
                    if len(inning_data_sorted) > 0:
                        row_to_add = inning_data_sorted.iloc[0].copy()
                        inning_data_sorted = inning_data_sorted.drop(inning_data_sorted.index[0])
                    else:
                        continue
                processed_regular.append(row_to_add)

        if processed_regular:
            regular_df = pd.DataFrame(processed_regular)
            regular_df = regular_df.drop('is_substitute', axis=1)
            result_rows.append(regular_df)
        if len(substitute_pas) > 0:
            substitute_pas = substitute_pas.drop('is_substitute', axis=1)
            result_rows.append(substitute_pas)

    if result_rows:
        return pd.concat(result_rows, ignore_index=True)
    else:
        return df


def reorder_승부주자(df):
    result_parts = []
    for team_name in df['team'].unique():
        team_df = df[df['team'] == team_name].copy()
        team_result = []
        for inning in sorted(team_df['inning'].unique()):
            inning_df = team_df[team_df['inning'] == inning].copy()
            승부_mask = is_승부주자(inning_df['result'])
            승부주자_rows = inning_df[승부_mask].copy()
            일반_rows = inning_df[~승부_mask].copy()
            if len(승부주자_rows) > 0:
                orders = set(승부주자_rows['order'].tolist())
                if orders == {1, 9}:
                    승부주자_rows = 승부주자_rows.sort_values('order', ascending=False)
                else:
                    승부주자_rows = 승부주자_rows.sort_values('order', ascending=True)
                inning_df = pd.concat([승부주자_rows, 일반_rows])
            team_result.append(inning_df)
        result_parts.append(pd.concat(team_result))
    return pd.concat(result_parts, ignore_index=True)


def extract_game_data(game_idx):
    url = f'https://www.korea-baseball.com/game/record_detail?game_idx={game_idx}'
    try:
        kst = pytz.timezone('Asia/Seoul')
        print(f"\n[{datetime.now(kst).strftime('%H:%M:%S')}] game_idx={game_idx} 데이터 가져오는 중...")
        headers = {'User-Agent': 'Mozilla/5.0'}

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = requests.get(url, headers=headers, timeout=30)
                break
            except requests.exceptions.ConnectionError as e:
                if attempt < max_retries - 1:
                    print(f"  연결 실패 ({attempt+1}/{max_retries}), 30초 후 재시도...")
                    time.sleep(30)
                else:
                    raise e

        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')

        if is_game_suspended(soup):
            tournament_elem = soup.find('dl', class_='game_name')
            if tournament_elem:
                date_info = tournament_elem.find('dd').get_text(strip=True)
                print(f"  ⚠ 일시정지 경기 감지: {date_info}")
            else:
                print(f"  ⚠ 일시정지 경기 감지")
            return "SUSPENDED"

        tournament_elem = soup.find('dl', class_='game_name')
        if not tournament_elem:
            print(f"  경고: game_idx={game_idx}에서 경기 정보를 찾을 수 없습니다.")
            return None

        tournament = tournament_elem.find('dt').get_text(strip=True)
        date_info = tournament_elem.find('dd').get_text(strip=True)
        parts = date_info.split()
        date_str = parts[0]
        time_str = parts[1]
        stadium = ' '.join(parts[2:]) if len(parts) > 2 else ''

        # 라운드(리그전/16강전/결승전)는 record_detail 에 없고 box_score 에만 있다.
        # 예전에는 크롤이 다 끝난 뒤 backfill_league_column 으로 한 바퀴 더 돌며 받아왔는데,
        # 그 패스가 경기마다 10초씩 또 쉬고 월별 엑셀을 전부 다시 읽고 썼다. 어차피
        # crawl_games 가 경기마다 10~12초를 쉬므로, 그 안에서 한 번 더 받는 편이 낫다
        # (서버로 나가는 요청 수는 경기당 2회로 그대로다 — 패스만 하나로 합친 것).
        time.sleep(random.uniform(1, 2))   # 같은 서버에 두 번 연속 때리지 않도록 간격
        league, box_stadium = fetch_game_name_dd(game_idx)
        if box_stadium:
            stadium = box_stadium          # 슬래시로 갈린 쪽이 안전하다(fetch_game_name_dd 주석)
        if not league:
            print(f"  ⚠ league 를 받지 못했습니다 — 경기는 그대로 저장하고 비워둡니다"
                  f" (나중에 backfill_league_column({game_idx}) 로 채울 것)")
        details = f"{date_str} {time_str} {stadium}" + (f" {league}" if league else "")
        year = int(date_str.split('.')[0])

        team1_elem = soup.find('div', class_='team1')
        team1 = team1_elem.find('span', class_='txt').get_text(strip=True)
        result1 = team1_elem.find('dd', class_='result').get_text(strip=True)
        score1 = int(team1_elem.find('dd', class_='score').get_text(strip=True))

        team2_elem = soup.find('div', class_='team2')
        team2 = team2_elem.find('span', class_='txt').get_text(strip=True)
        result2 = team2_elem.find('dd', class_='result').get_text(strip=True)
        score2 = int(team2_elem.find('dd', class_='score').get_text(strip=True))

        print(f"  경기: {team1} vs {team2} ({date_str})")

        tables = soup.find_all('table', class_='record_table')
        team1_pitchers, team2_pitchers = extract_pitcher_info(soup, tables)
        print(f"  투수 정보: team1={len(team1_pitchers)}명, team2={len(team2_pitchers)}명")

        if len(tables) >= 8:
            타자_테이블_인덱스 = [0, 4]
        elif len(tables) >= 4:
            타자_테이블_인덱스 = [0, 2]
        else:
            타자_테이블_인덱스 = [0, min(2, len(tables)-1)] if len(tables) >= 2 else [0]

        topbot_구분 = ['top', 'bot']
        팀_구분 = [team1, team2]
        all_data = []

        for table_num, topbot, team in zip(타자_테이블_인덱스, topbot_구분, 팀_구분):
            if table_num >= len(tables):
                continue
            table = tables[table_num]
            tbody = table.find('tbody')
            if not tbody:
                continue

            data = []
            for tr in tbody.find_all('tr'):
                if 'sum' in tr.get('class', []):
                    continue
                row = []
                for th in tr.find_all('th'):
                    row.append(th.get_text(strip=True))
                for td in tr.find_all('td'):
                    row.append(td.get_text(strip=True))
                data.append(row)

            thead = table.find('thead')
            if not thead:
                continue

            headers_list = [th.get_text(strip=True) for th in thead.find('tr').find_all('th')]
            df = pd.DataFrame(data, columns=headers_list)
            if df.empty:
                continue

            이닝_컬럼 = [str(i) for i in range(1, 21)]
            타석기록 = df.melt(
                id_vars=['타순', '포지션', '선수명'],
                value_vars=이닝_컬럼,
                var_name='이닝',
                value_name='기록'
            )
            타석기록 = 타석기록[타석기록['기록'].str.strip() != ''].copy()
            타석기록['이닝'] = 타석기록['이닝'].astype(int)
            타석기록['타순'] = 타석기록['타순'].astype(int)
            타석기록 = 타석기록.sort_values(['이닝', '타순']).reset_index(drop=True)

            expanded_rows = []
            for _, row in 타석기록.iterrows():
                기록_텍스트 = str(row['기록'])
                if '/' in 기록_텍스트:
                    for 타석_기록 in 기록_텍스트.split('/'):
                        new_row = row.copy()
                        new_row['기록'] = 타석_기록.strip()
                        expanded_rows.append(new_row)
                else:
                    expanded_rows.append(row)

            타석기록 = pd.DataFrame(expanded_rows).reset_index(drop=True)
            타석기록['topbot'] = topbot
            타석기록['team'] = team
            타석기록['seq'] = range(1, len(타석기록) + 1)
            all_data.append(타석기록)

        if not all_data:
            print(f"  경고: game_idx={game_idx}에서 타자 기록을 찾을 수 없습니다.")
            return None

        전체타석기록 = pd.concat(all_data, ignore_index=True)
        전체타석기록['def_team'] = 전체타석기록.apply(
            lambda row: team2 if row['topbot'] == 'top' else team1, axis=1
        )
        전체타석기록['pitcher'] = ''
        전체타석기록['game_id'] = game_idx
        전체타석기록['tournament'] = tournament
        전체타석기록['details'] = details
        전체타석기록['year'] = year
        전체타석기록['date'] = date_str
        전체타석기록['time'] = time_str
        전체타석기록['stadium'] = stadium
        전체타석기록['league'] = league if league else ''
        전체타석기록['team1'] = team1
        전체타석기록['score1'] = score1
        전체타석기록['result1'] = result1
        전체타석기록['team2'] = team2
        전체타석기록['score2'] = score2
        전체타석기록['result2'] = result2

        최종 = 전체타석기록[[
            'game_id', 'team', 'seq', 'topbot', '이닝', '타순', '선수명', '포지션', '기록',
            'tournament', 'details', 'year', 'date', 'time', 'stadium', 'league',
            'team1', 'score1', 'result1', 'team2', 'score2', 'result2',
            'def_team', 'pitcher'
        ]].copy()

        최종 = apply_batting_order_continuity(최종)

        최종.columns = [
            'game_id', 'team', 'plate_seq', 'topbot', 'inning', 'order', 'batter', 'position', 'result',
            'tournament', 'details', 'year', 'date', 'time', 'stadium', 'league',
            'team1', 'score1', 'result1', 'team2', 'score2', 'result2',
            'def_team', 'pitcher'
        ]

        최종.insert(7, 'on_base_123', '')
        최종.insert(9, 'outs_when_up', '')
        최종.insert(10, 'away_score', '')
        최종.insert(11, 'home_score', '')

        # 땅R, 앞R 등 타자는 살고 주자가 아웃되는 예외 케이스에서
        # 어느 타순이 죽었는지 수기로 기록하기 위한 열 (수집 시점에는 항상 빈 칸)
        최종['예외_아웃'] = ''

        # 타점 (수집 시점에는 항상 빈 칸, 이후 별도 계산 로직으로 채움)
        최종['RBI'] = ''

        def calculate_outs(result):
            """result 에서 '이 행이 만든 아웃 개수'를 추정해 미리 채운다.
            검수 단계에서 사람이 확인·수정하는 값이라, 확실한 것만 채우고 애매하면 빈칸으로 둔다.

            쉼표로 이어진 결과는 조각별로 본다. 통짜 문자열에 부분일치를 걸면
            '유직,포일'(앞 조각이 아웃)을 놓치고 '투번안,송구실책'(앞 조각이 안타)을
            잘못 잡는다.
            """
            result_str = str(result)
            if '병살' in result_str:
                return 2
            parts = [p.strip() for p in result_str.split(',')]
            # 직선타 아웃 — '직'으로 끝나는 조각(유직·투직·1직…).
            #   '3직실'(직선타를 야수 실책으로 출루)은 '직'으로 끝나지 않아 저절로 빠지고,
            #   '1직병살'은 위의 병살 검사에서 이미 2로 잡힌다.
            if any(p.endswith('직') for p in parts):
                return 1
            # 낫아웃- : 타자가 1루에서 잡힌 것 = 타자 아웃.
            #   '낫아웃+'(타자가 살아 나감)와 반대라 부호로만 갈린다. 검수 파이프라인도
            #   같은 방식으로 본다(_clean_result 의 `'낫아웃-' in natout`,
            #   _process_r_result 의 batter_safe 판정).
            if any('낫아웃-' in p for p in parts):
                return 1
            # 타자는 살고 주자가 아웃된 예외 케이스 — 결과 끝에 'R'이 붙는다
            # (유땅R·3땅R·투번R·투앞R·중앞R…). 어느 타순이 죽었는지는 '예외_아웃' 열에
            # 수기로 적지만, 아웃이 하나 났다는 것 자체는 확실하다.
            #   '땅'·'번'이 든 형태는 아래 검사에도 걸리지만 '앞R'은 안 걸린다 —
            #   그래서 접미사 R 자체를 본다.
            #   'R[포구실책]'·'R[송구실책]'은 아무도 안 죽은 것이라(검수 파이프라인의
            #   _process_r_result 도 안타로 처리한다) 'R'로 끝나지 않아 저절로 빠지고,
            #   'R[병살타]'는 위의 병살 검사에서 이미 2로 잡힌다.
            if any(p.endswith('R') for p in parts):
                return 1
            # 타자가 살아 나간 조각은 이 행이 만든 아웃이 아니다 — 매칭에서 빼낸다.
            #   · '번안'(투번안·3번안·포번안…)   = 번트 안타
            #   · '번출'(투희번출·1희번출…)      = 번트 출루
            #   · '플출'(좌희플출·중희플출…)      = 뜬공 출루. 공이 안 잡혔다는 뜻이다
            #   · '실'로 끝남(유땅실·2땅실·좌플실…) = 야수 실책으로 출루
            #   야수선택[FC]은 앞 주자를 '못 잡았을 때' 기록하는 것이라 아무도 죽지 않고,
            #   송구·포구실책이 붙은 경우도 마찬가지다.
            #   희생번트(투희번)·희생플라이(중희플)·번트 아웃(투번)·쓰리번트·평범한
            #   땅볼(유땅)은 어디에도 안 걸려 그대로 1로 남는다. '송구실책'·'포구실책'은
            #   '책'으로 끝나므로 '실' 검사에 걸리지 않는다.
            #   '투희번출,주자아웃' 처럼 주자가 죽은 경우의 아웃은 이 행 몫이 아니라
            #   '주자아웃' 후속행(또는 '예외_아웃' 열) 몫이므로 여기서 세지 않는다.
            if any(any(k in p for k in ['땅', '플', '번', '삼진'])
                   and not any(x in p for x in ('번안', '번출', '플출'))
                   and not p.endswith('실')
                   for p in parts):
                return 1
            return ''

        최종['outs_when_up'] = 최종['result'].apply(calculate_outs)

        최종 = 최종[[
            'game_id', 'team', 'plate_seq', 'inning', 'topbot', 'order', '예외_아웃', 'batter',
            'on_base_123', 'result', 'outs_when_up', 'away_score', 'home_score', 'RBI', 'position',
            'tournament', 'details', 'year', 'date', 'time', 'stadium', 'league',
            'team1', 'score1', 'result1', 'team2', 'score2', 'result2',
            'def_team', 'pitcher'
        ]]

        대체_rows = 최종[is_pure_substitute(최종['result'])].copy()
        일반_rows = 최종[~is_pure_substitute(최종['result'])].copy()
        일반_rows = reorder_승부주자(일반_rows)
        최종 = pd.concat([일반_rows, 대체_rows], ignore_index=True)

        최종['plate_seq'] = 최종['plate_seq'].astype(object)
        for team_name in [team1, team2]:
            team_mask = 최종['team'] == team_name
            team_data = 최종[team_mask].copy()
            is_regular = ~is_pure_substitute(team_data['result'])
            seq_counter = 1
            new_seqs = []
            for _, is_reg in is_regular.items():
                if is_reg:
                    new_seqs.append(seq_counter)
                    seq_counter += 1
                else:
                    new_seqs.append('')
            최종.loc[team_mask, 'plate_seq'] = new_seqs

        top_indices = 최종[(최종['topbot'] == 'top') & (~is_pure_substitute(최종['result']))].index
        if len(team2_pitchers) >= len(top_indices):
            최종.loc[top_indices, 'pitcher'] = team2_pitchers[:len(top_indices)]
        elif len(team2_pitchers) > 0:
            pitchers_repeated = (team2_pitchers * ((len(top_indices) // len(team2_pitchers)) + 1))[:len(top_indices)]
            최종.loc[top_indices, 'pitcher'] = pitchers_repeated

        bot_indices = 최종[(최종['topbot'] == 'bot') & (~is_pure_substitute(최종['result']))].index
        if len(team1_pitchers) >= len(bot_indices):
            최종.loc[bot_indices, 'pitcher'] = team1_pitchers[:len(bot_indices)]
        elif len(team1_pitchers) > 0:
            pitchers_repeated = (team1_pitchers * ((len(bot_indices) // len(team1_pitchers)) + 1))[:len(bot_indices)]
            최종.loc[bot_indices, 'pitcher'] = pitchers_repeated

        print(f"  완료: {len(최종)}개 기록 추출")
        return 최종

    except Exception as e:
        print(f"  오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return None


# ==================== A안: 자동 재시도 ====================
def retry_auto(log):
    failed_ids = log.get("failed_game_ids", [])
    if not failed_ids:
        return

    print(f"\n[A안] 이전 실패 game_id {len(failed_ids)}개 자동 재시도...")
    success_list, still_fail_list, suspended_list = crawl_games(failed_ids)

    if success_list:
        success_list.sort(key=lambda x: x[1]["date"].iloc[0])
        save_games_to_files(success_list, log)
        run_post_processing(success_list)

    if suspended_list:
        existing_suspended = set(log.get("suspended_game_ids", []))
        log["suspended_game_ids"] = list(existing_suspended | set(suspended_list))
        still_fail_list = [gid for gid in still_fail_list if gid not in suspended_list]
        print(f"[A안] 일시정지 경기 {len(suspended_list)}개 → suspended_game_ids로 이동: {suspended_list}")

    log["failed_game_ids"] = still_fail_list
    save_log(log)

    print(f"[A안] 재시도 결과 → 성공: {len(success_list)} / 일시정지: {len(suspended_list)} / 여전히 실패: {len(still_fail_list)}")
    if still_fail_list:
        print(f"  여전히 실패한 game_id: {still_fail_list}")


# ==================== B안 도우미: 지정 파일 기준 저장 (10경기 초과 / 월 불일치 자동 분기) ====================
def _append_to_target_file(new_game_data_list, target_file_path, log):
    """
    target_file_path로 지정된 파일의 월에 속하는 경기는 그 파일부터 이어붙이되,
    10경기가 차면 save_games_to_files()와 동일하게 같은 월 폴더 안에서 다음
    알파벳 파일로 넘어간다.

    target_file_path의 월과 실제 경기 날짜의 월이 다르면(재처리 대상에 여러
    달이 섞여 있는 경우) 그 경기들은 save_games_to_files()로 넘겨 해당 월
    폴더/파일을 자동으로 찾거나 새로 만들어 저장한다.
    """
    target_file_path = unicodedata.normalize('NFC', target_file_path)
    target_fname = os.path.basename(target_file_path)
    target_month_key, target_letter = os.path.splitext(target_fname)[0].rsplit('_', 1)

    same_month_games = []
    other_month_games = []
    for game_id, df in new_game_data_list:
        date_str = df["date"].iloc[0]
        if get_month_key(date_str) == target_month_key:
            same_month_games.append((game_id, df))
        else:
            other_month_games.append((game_id, df))

    if same_month_games:
        if os.path.exists(target_file_path):
            pending_df = load_existing_file(target_file_path)
            current_count = pending_df["game_id"].nunique() if not pending_df.empty else 0
        else:
            pending_df = pd.DataFrame()
            current_count = 0

        current_letter = target_letter
        current_file_path = target_file_path

        while current_count >= GAMES_PER_FILE:
            current_letter = next_letter(current_letter)
            current_file_path = get_file_path(target_month_key, current_letter)
            if os.path.exists(current_file_path):
                pending_df = load_existing_file(current_file_path)
                current_count = pending_df["game_id"].nunique() if not pending_df.empty else 0
            else:
                pending_df = pd.DataFrame()
                current_count = 0

        for game_id, game_df in same_month_games:
            pending_df = pd.concat([pending_df, game_df], ignore_index=True)
            current_count += 1

            if current_count >= GAMES_PER_FILE:
                pending_df.to_excel(current_file_path, index=False, engine="openpyxl")
                print(f"  저장 완료: {os.path.basename(current_file_path)} ({current_count}경기)")
                current_letter = next_letter(current_letter)
                current_file_path = get_file_path(target_month_key, current_letter)
                pending_df = pd.DataFrame()
                current_count = 0

        if not pending_df.empty:
            pending_df.to_excel(current_file_path, index=False, engine="openpyxl")
            print(f"  저장 완료: {os.path.basename(current_file_path)} ({current_count}경기, 미완료)")

        month_file_status = log.get("month_file_status", {})
        month_file_status[target_month_key] = {
            "last_file": current_letter,
            "game_count": current_count
        }
        log["month_file_status"] = month_file_status
        save_log(log)

    if other_month_games:
        print(f"  지정 파일({target_month_key})과 다른 월 경기 {len(other_month_games)}개 감지 → 해당 월 폴더에 자동 저장")
        other_month_games.sort(key=lambda x: x[1]["date"].iloc[0])
        save_games_to_files(other_month_games, log)


# ==================== B안: 수동 재처리 (특정 파일 지정 또는 월별 자동 저장) ====================
def retry_failed_games(failed_game_ids, target_file_path=None):
    """
    실패한 game_id 목록을 직접 입력받아 재처리
    target_file_path: 지정 시 해당 xlsx 파일에 append, None이면 기존 로직대로 월별 자동 저장
    성공한 game_id는 log에서 failed_game_ids 제거 + collected_ids에 추가
    """
    log = load_log()

    print(f"[B안] 재처리 대상: {len(failed_game_ids)}경기")
    if target_file_path:
        print(f"저장 대상 파일: {target_file_path}")

    new_game_data_list = []
    still_fail_list = []

    for i, game_id in enumerate(failed_game_ids, 1):
        print(f"\n[{i}/{len(failed_game_ids)}] game_idx={game_id} 재시도...")
        df = extract_game_data(game_id)

        if df is not None:
            new_game_data_list.append((game_id, df))
        else:
            still_fail_list.append(game_id)
            print(f"  재시도 실패: game_idx={game_id}")

        if i < len(failed_game_ids):
            delay = random.uniform(10, 12)
            print(f"  {delay:.1f}초 대기...")
            time.sleep(delay)

    if new_game_data_list:
        if target_file_path:
            # ── 지정 파일부터 이어붙이되, 10경기 초과 / 월 불일치는 자동 분기 ──
            _append_to_target_file(new_game_data_list, target_file_path, log)

        else:
            # ── 기존 로직: 날짜 기준 월별 자동 저장 ──
            new_game_data_list.sort(key=lambda x: x[1]["date"].iloc[0])
            save_games_to_files(new_game_data_list, log)

        run_post_processing(new_game_data_list)

        # ── log 업데이트 (저장 방식 무관하게 공통 처리) ──
        succeeded_ids = [game_id for game_id, _ in new_game_data_list]

        # failed_game_ids에서 성공한 것 제거
        log["failed_game_ids"] = [gid for gid in log.get("failed_game_ids", []) if gid not in succeeded_ids]

        # collected_ids에 성공한 것 추가 (중복 방지)
        existing_collected = set(log.get("collected_ids", []))
        log["collected_ids"] = list(existing_collected | set(succeeded_ids))

        # game_ids(= run_pipeline 일반 모드의 중복 제거 기준)에도 넣어 둔다.
        # 여기 없으면 나중에 run_pipeline()이 같은 경기를 '신규'로 다시 잡는다.
        existing_game_ids = log.get("game_ids", [])
        log["game_ids"] = existing_game_ids + [
            gid for gid in succeeded_ids if gid not in existing_game_ids
        ]

        save_log(log)
        print(f"  log 업데이트 완료: {succeeded_ids} → failed_game_ids 제거, collected_ids 추가")

    print("\n" + "=" * 60)
    print(f"[B안] 완료! 성공: {len(new_game_data_list)} / 실패: {len(still_fail_list)}")
    if still_fail_list:
        print(f"여전히 실패한 game_id: {still_fail_list}")
    print("=" * 60)

# ==================== C안: 일시정지 경기 재시도 ====================
def retry_suspended_games(target_file_path=None):
    log = load_log()
    suspended_ids = log.get("suspended_game_ids", [])

    if not suspended_ids:
        print("재시도할 일시정지 경기가 없습니다.")
        return

    print(f"[C안] 일시정지 경기 재시도: {len(suspended_ids)}경기 → {suspended_ids}")

    success_list, fail_list, still_suspended_list = crawl_games(suspended_ids)

    if success_list:
        if target_file_path:
            new_df = pd.concat([df for _, df in success_list], ignore_index=True)
            target_file_path = unicodedata.normalize('NFC', target_file_path)

            if os.path.exists(target_file_path):
                existing_df = pd.read_excel(target_file_path)
                combined_df = pd.concat([existing_df, new_df], ignore_index=True)
                print(f"  기존 파일에 append: {len(existing_df)}행 → {len(combined_df)}행")
            else:
                combined_df = new_df
                print(f"  파일 없음 → 새로 생성: {target_file_path}")

            combined_df.to_excel(target_file_path, index=False)
            print(f"  저장 완료: {target_file_path}")
        else:
            # target_file_path 없으면 월별 마지막 파일에 append
            month_groups = defaultdict(list)
            for game_id, df in success_list:
                date_str = df["date"].iloc[0]
                month_key = get_month_key(date_str)
                month_groups[month_key].append((game_id, df))

            for month_key, games in sorted(month_groups.items()):
                new_df = pd.concat([df for _, df in games], ignore_index=True)
                last_file = get_last_file_in_month(month_key)

                if last_file:
                    existing_df = pd.read_excel(last_file, engine="openpyxl")
                    combined_df = pd.concat([existing_df, new_df], ignore_index=True)
                    combined_df.to_excel(last_file, index=False, engine="openpyxl")
                    print(f"  append 완료: {os.path.basename(last_file)} ({len(existing_df)}행 → {len(combined_df)}행)")
                else:
                    first_file = get_file_path(month_key, "a")
                    new_df.to_excel(first_file, index=False, engine="openpyxl")
                    print(f"  새 파일 생성: {os.path.basename(first_file)}")

        succeeded_ids = [game_id for game_id, _ in success_list]
        existing_collected = set(log.get("collected_ids", []))
        log["collected_ids"] = list(existing_collected | set(succeeded_ids))

        # game_ids(= run_pipeline 일반 모드의 중복 제거 기준)에도 넣어 둔다.
        existing_game_ids = log.get("game_ids", [])
        log["game_ids"] = existing_game_ids + [
            gid for gid in succeeded_ids if gid not in existing_game_ids
        ]
        print(f"  log 업데이트: {succeeded_ids} → suspended 제거, collected_ids/game_ids 추가")

        run_post_processing(success_list)

    remaining_suspended = list(set(still_suspended_list))
    log["suspended_game_ids"] = remaining_suspended

    if fail_list:
        existing_failed = set(log.get("failed_game_ids", []))
        log["failed_game_ids"] = list(existing_failed | set(fail_list))
        print(f"  파싱 실패 {len(fail_list)}개 → failed_game_ids로 이동: {fail_list}")

    save_log(log)

    print("\n" + "=" * 60)
    print(f"[C안] 완료 → 성공: {len(success_list)} / 여전히 일시정지: {len(remaining_suspended)} / 파싱실패: {len(fail_list)}")
    if remaining_suspended:
        print(f"  여전히 일시정지 (suspended_game_ids 유지): {remaining_suspended}")
    print("=" * 60)


# ==================== 투수 박스스코어 파싱 ====================
def parse_ip(ip_str):
    try:
        ip_str = str(ip_str).strip()
        if '.' in ip_str:
            full, partial = ip_str.split('.')
            return int(full) + int(partial) / 3
        else:
            return float(ip_str)
    except:
        return None


def parse_pitcher_box(game_idx, session=None):
    url = f"https://www.korea-baseball.com/game/record_detail?game_idx={game_idx}"

    try:
        if session:
            res = session.get(url, timeout=10)
        else:
            res = requests.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{game_idx}] 요청 실패: {e}")
        return []

    soup = BeautifulSoup(res.text, 'html.parser')
    records = []

    hero = soup.select('.hero_sum .team')
    team1 = hero[0].select_one('.txt').get_text(strip=True) if len(hero) > 0 else None
    team2 = hero[1].select_one('.txt').get_text(strip=True) if len(hero) > 1 else None

    sections = soup.select('.section_sumrec')
    for section in sections:
        h4 = section.select_one('h4')
        if not h4 or '투수기록' not in h4.get_text():
            continue

        def_team = h4.get_text(strip=True).replace('투수기록', '').strip()
        table = section.select_one('table[summary="투수기록"]')
        if not table:
            continue

        rows = table.select('tbody tr')
        for row in rows:
            if 'sum' in row.get('class', []):
                continue
            cells = row.find_all(['th', 'td'])
            if len(cells) < 16:
                continue
            try:
                record = {
                    'game_idx': game_idx,
                    'team1':    team1,
                    'team2':    team2,
                    'def_team': def_team,
                    'pitcher':  cells[0].get_text(strip=True),
                    'entry':    cells[1].get_text(strip=True),
                    'result':   cells[2].get_text(strip=True),
                    'W':        cells[3].get_text(strip=True),
                    'L':        cells[4].get_text(strip=True),
                    'IP_raw':   cells[5].get_text(strip=True),
                    'BF':       cells[6].get_text(strip=True),
                    'pitches':  cells[7].get_text(strip=True),
                    'AB':       cells[8].get_text(strip=True),
                    'H':        cells[9].get_text(strip=True),
                    'HR':       cells[10].get_text(strip=True),
                    'BB_HBP':   cells[11].get_text(strip=True),
                    'K':        cells[12].get_text(strip=True),
                    'R':        cells[13].get_text(strip=True),
                    'ER':       cells[14].get_text(strip=True),
                    'ERA':      cells[15].get_text(strip=True),
                }
                records.append(record)
            except Exception as e:
                print(f"  [{game_idx}] 행 파싱 오류: {e}")
                continue

    return records


def _save_pitcher_box(records, output_path):
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df['IP'] = df['IP_raw'].apply(parse_ip)

    int_cols = ['W', 'L', 'BF', 'pitches', 'AB', 'H', 'HR', 'BB_HBP', 'K', 'R', 'ER']
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    df['ERA'] = pd.to_numeric(df['ERA'], errors='coerce')

    col_order = ['game_idx', 'team1', 'team2', 'def_team', 'pitcher',
                 'entry', 'result', 'W', 'L', 'IP_raw', 'IP',
                 'BF', 'pitches', 'AB', 'H', 'HR', 'BB_HBP', 'K', 'R', 'ER', 'ERA']
    df = df[[c for c in col_order if c in df.columns]]
    df.to_excel(output_path, index=False)
    return df


def crawl_pitcher_box_by_year(game_idx_by_year, output_dir, save_every=50, max_retry=3):
    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })

    total_failed = {}

    for year, game_idx_list in sorted(game_idx_by_year.items()):
        output_path = os.path.join(output_dir, f'pitcher_box_{year}.xlsx')

        if os.path.exists(output_path):
            existing_df = pd.read_excel(output_path)
            already_collected = set(existing_df['game_idx'].dropna().astype(int).tolist())
            all_records = existing_df.to_dict('records')
            print(f"\n{year}년 기존 파일 로드: {len(existing_df)}행, {len(already_collected)}경기 수집됨")
        else:
            already_collected = set()
            all_records = []

        remaining = [gid for gid in game_idx_list if gid not in already_collected]

        print(f"\n{'='*50}")
        print(f"{year}년 투수 박스스코어 수집 ({len(remaining)}경기 신규 / 전체 {len(game_idx_list)}경기, {len(already_collected)}경기 스킵)")
        print(f"{'='*50}")

        if not remaining:
            print(f"{year}년 새로 수집할 경기 없음, 스킵")
            total_failed[year] = []
            continue

        failed = []

        for i, game_idx in enumerate(remaining):
            records = []
            elapsed = 0
            for attempt in range(1, max_retry + 1):
                start = time.time()
                records = parse_pitcher_box(game_idx, session)
                elapsed = time.time() - start

                if records:
                    print(f"  [{i+1}/{len(remaining)}] {game_idx} → {len(records)}명 ({elapsed:.1f}s)")
                    break
                else:
                    if attempt < max_retry:
                        retry_wait = random.uniform(9.0, 11.0)
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 실패 ({elapsed:.1f}s) / 재시도 {attempt}/{max_retry} ({retry_wait:.1f}s 후)")
                        time.sleep(retry_wait)
                    else:
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 최종 실패 ({max_retry}회 시도)")
                        failed.append(game_idx)

            if records:
                all_records.extend(records)

            if (i + 1) % save_every == 0:
                _save_pitcher_box(all_records, output_path)
                print(f"  >>> 중간 저장 완료 ({i+1}/{len(remaining)})")

            target = random.uniform(9.0, 11.0)
            wait = max(0, target - elapsed)
            time.sleep(wait)

        df = _save_pitcher_box(all_records, output_path)
        total_failed[year] = failed

        print(f"\n{year}년 완료: {len(df)}행 저장")
        print(f"실패 {len(failed)}개: {failed if failed else '없음'}")

    return total_failed


# ==================== 타자 박스스코어 파싱 ====================
def parse_batter_box(game_idx, session=None):
    url = f"https://www.korea-baseball.com/game/record_detail?game_idx={game_idx}"

    try:
        if session:
            resp = session.get(url, timeout=10)
        else:
            resp = requests.get(url, timeout=10)
        resp.raise_for_status()
    except Exception as e:
        print(f"  [{game_idx}] 요청 실패: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    sections = soup.find_all("div", class_="section_sumrec")
    records = []

    # 타자기록 섹션 순서 카운터 (0=초/원정, 1=말/홈)
    batter_section_idx = 0

    for section in sections:
        h4 = section.find("h4")
        if h4 is None:
            continue
        h4_text = h4.get_text(strip=True)
        if "타자기록" not in h4_text:
            continue

        team = h4_text.replace("타자기록", "").strip()
        topbot = "top" if batter_section_idx == 0 else "bot"
        batter_section_idx += 1

        # tb_body 우선, 없으면 summary="타자기록" 첫 번째 테이블 사용
        table = None
        tb_wr = section.find("div", class_="tb_wr")
        if tb_wr:
            tb_body = tb_wr.find("div", class_="tb_body")
            if tb_body:
                table = tb_body.find("table")

        if table is None:
            table = section.find("table", summary="타자기록")

        if table is None:
            continue

        rows = table.find("tbody").find_all("tr")

        for row in rows:
            if "sum" in row.get("class", []):
                continue

            ths = row.find_all("th")
            if len(ths) < 3:
                continue

            batting_order_text = ths[0].get_text(strip=True)
            try:
                batting_order = int(batting_order_text)
            except ValueError:
                continue

            pos_span = ths[1].find("span", class_="position")
            position = pos_span.get_text(separator="/", strip=True) if pos_span else ths[1].get_text(strip=True)

            player_raw = ths[2].get_text(strip=True)
            # 등번호가 비어 있는 선수가 있다("김동영()"). (\d+) 로는 매칭이 안 돼 이름에
            # 괄호가 그대로 남고, 그러면 season_full·선수 마스터와 이름이 안 붙는다
            # (2026 batter_box 22,034행 중 429행이 그랬다). 투수기록 쪽 _strip_name_no 와
            # 같은 규칙으로 빈 번호까지 떼어낸다.
            match = re.match(r'^(.*?)\((\d*)\D*$', player_raw)
            if match:
                player_name = match.group(1).strip()
                player_num = int(match.group(2)) if match.group(2) else None
            else:
                player_name = player_raw.strip()
                player_num = None

            tds = row.find_all("td")
            inning_cells = [td for td in tds if "fixed" not in td.get("class", [])]
            inning_texts = [td.get_text(strip=True) for td in inning_cells]
            is_pinch_hitter = any("대타" in t for t in inning_texts)

            fixed_tds = [td for td in tds if "fixed" in td.get("class", [])]
            numeric_fixed = []
            for td in fixed_tds:
                text = td.get_text(strip=True)
                try:
                    int(text)
                    numeric_fixed.append(text)
                except ValueError:
                    pass

            if len(numeric_fixed) < 4:
                continue

            try:
                ab  = int(numeric_fixed[0])
                h   = int(numeric_fixed[1])
                rbi = int(numeric_fixed[2])
                r   = int(numeric_fixed[3])
            except ValueError:
                continue

            records.append({
                "game_idx":        game_idx,
                "topbot":          topbot,
                "team":            team,
                "batting_order":   batting_order,
                "player_name":     player_name,
                "player_num":      player_num,
                "position":        position,
                "is_pinch_hitter": is_pinch_hitter,
                "AB":              ab,
                "H":               h,
                "RBI":             rbi,
                "R":               r,
            })

    return records


def _save_batter_box(all_records, output_path):
    if not all_records:
        print("  [저장 스킵] 데이터 없음")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_excel(output_path, index=False)
    print(f"  [저장] {output_path} ({len(df)}행)")
    return df


def crawl_batter_box_by_year(game_idx_by_year, output_dir, save_every=50, max_retry=3):
    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })

    total_failed = {}

    for year, game_idx_list in sorted(game_idx_by_year.items()):
        output_path = os.path.join(output_dir, f'batter_box_{year}.xlsx')

        if os.path.exists(output_path):
            existing_df = pd.read_excel(output_path)
            already_collected = set(existing_df['game_idx'].dropna().astype(int).tolist())
            all_records = existing_df.to_dict('records')
            print(f"\n{year}년 기존 파일 로드: {len(existing_df)}행, {len(already_collected)}경기 수집됨")
        else:
            already_collected = set()
            all_records = []

        remaining = [gid for gid in game_idx_list if gid not in already_collected]

        print(f"\n{'='*50}")
        print(f"{year}년 타자 박스스코어 수집 ({len(remaining)}경기 신규 / 전체 {len(game_idx_list)}경기, {len(already_collected)}경기 스킵)")
        print(f"{'='*50}")

        if not remaining:
            print(f"{year}년 새로 수집할 경기 없음, 스킵")
            total_failed[year] = []
            continue

        failed = []

        for i, game_idx in enumerate(remaining):
            records = []
            elapsed = 0
            for attempt in range(1, max_retry + 1):
                start = time.time()
                records = parse_batter_box(game_idx, session)
                elapsed = time.time() - start

                if records:
                    print(f"  [{i+1}/{len(remaining)}] {game_idx} → {len(records)}명 ({elapsed:.1f}s)")
                    break
                else:
                    if attempt < max_retry:
                        retry_wait = random.uniform(9.0, 11.0)
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 실패 ({elapsed:.1f}s) / 재시도 {attempt}/{max_retry} ({retry_wait:.1f}s 후)")
                        time.sleep(retry_wait)
                    else:
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 최종 실패 ({max_retry}회 시도)")
                        failed.append(game_idx)

            if records:
                all_records.extend(records)

            if (i + 1) % save_every == 0:
                _save_batter_box(all_records, output_path)
                print(f"  >>> 중간 저장 완료 ({i+1}/{len(remaining)})")

            target = random.uniform(9.0, 11.0)
            wait = max(0, target - elapsed)
            time.sleep(wait)

        df = _save_batter_box(all_records, output_path)
        total_failed[year] = failed

        print(f"\n{year}년 완료: {len(df)}행 저장")
        print(f"실패 {len(failed)}개: {failed if failed else '없음'}")

    return total_failed


# ==================== 이닝 스코어(점수) 파싱 ====================
def parse_inning_summary(game_idx, session=None):
    url = f"https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"

    try:
        if session:
            res = session.get(url, timeout=10)
        else:
            res = requests.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{game_idx}] 요청 실패: {e}")
        return []

    soup = BeautifulSoup(res.text, 'html.parser')

    # ── 팀명 파싱
    hero = soup.select('.hero_sum .team')
    team1 = hero[0].select_one('.txt').get_text(strip=True) if len(hero) > 0 else None
    team2 = hero[1].select_one('.txt').get_text(strip=True) if len(hero) > 1 else None

    if not team1 or not team2:
        print(f"  [{game_idx}] 팀명 파싱 실패")
        return []

    # ── 이닝 테이블: tb_head(신) → box_table(구) 순으로 시도
    table = soup.select_one('.tb_head table')
    if not table:
        table = soup.select_one('.box_table table')
    if not table:
        print(f"  [{game_idx}] 이닝 테이블 없음")
        return []

    rows = table.select('tbody tr')
    if len(rows) < 2:
        print(f"  [{game_idx}] tbody 행 부족 ({len(rows)}행)")
        return []

    records = []

    for row_idx, row in enumerate(rows[:2]):  # top/bot 두 행만
        topbot = 'top' if row_idx == 0 else 'bot'
        team   = team1 if row_idx == 0 else team2

        cells = row.find_all(['th', 'td'])
        if len(cells) < 6:
            continue

        inning_cells = cells[1:-4]   # 이닝 셀 (최대 20개)
        rheb_cells   = cells[-4:]    # R, H, E, B

        def parse_int(cell):
            txt = re.sub(r'[^\d]', '', cell.get_text(strip=True))
            return int(txt) if txt else 0

        R = parse_int(rheb_cells[0])
        H = parse_int(rheb_cells[1])
        E = parse_int(rheb_cells[2])
        B = parse_int(rheb_cells[3])

        # 이닝별 점수: 빈 셀은 0으로, 연속 2개 빈 셀이면 중단
        empty_count = 0
        for inning_num, cell in enumerate(inning_cells, start=1):
            txt = cell.get_text(strip=True)
            if txt == '':
                empty_count += 1
                if empty_count >= 2:
                    break
                continue
            else:
                empty_count = 0

            score = int(txt) if txt.isdigit() else 0

            records.append({
                'game_id': game_idx,
                'team1':   team1,
                'team2':   team2,
                'team':    team,
                'topbot':  topbot,
                'inning':  inning_num,
                'score':   score,
                'real_out': 3,
                'R':       R,
                'H':       H,
                'E':       E,
                'B':       B,
            })

    return records


def _save_inning_summary(records, output_path):
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    col_order = ['game_id', 'team1', 'team2', 'team', 'topbot',
                 'inning', 'score', 'real_out', 'R', 'H', 'E', 'B']
    df = df[[c for c in col_order if c in df.columns]]

    df.to_excel(output_path, index=False)
    return df


def crawl_inning_summary_by_year(game_idx_by_year, output_dir, save_every=50, max_retry=3):
    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })

    total_failed = {}

    for year, game_idx_list in sorted(game_idx_by_year.items()):
        output_path = os.path.join(output_dir, f'inning_summary_{year}.xlsx')

        # 기존 파일이 있으면 로드 후 이미 수집된 game_id는 스킵 (단, 이 파일의 실제 컬럼명은 game_id)
        if os.path.exists(output_path):
            existing_df = pd.read_excel(output_path)
            already_collected = set(existing_df['game_id'].dropna().astype(int).tolist())
            all_records = existing_df.to_dict('records')
            print(f"\n{year}년 기존 파일 로드: {len(existing_df)}행, {len(already_collected)}경기 수집됨")
        else:
            already_collected = set()
            all_records = []

        remaining = [gid for gid in game_idx_list if gid not in already_collected]

        print(f"\n{'='*50}")
        print(f"{year}년 이닝 스코어 수집 ({len(remaining)}경기 신규 / 전체 {len(game_idx_list)}경기, {len(already_collected)}경기 스킵)")
        print(f"{'='*50}")

        if not remaining:
            print(f"{year}년 새로 수집할 경기 없음, 스킵")
            total_failed[year] = []
            continue

        failed = []

        for i, game_idx in enumerate(remaining):

            records = []
            elapsed = 0
            for attempt in range(1, max_retry + 1):
                start = time.time()
                records = parse_inning_summary(game_idx, session)
                elapsed = time.time() - start

                if records:
                    print(f"  [{i+1}/{len(remaining)}] {game_idx} → {len(records)}행 ({elapsed:.1f}s)")
                    break
                else:
                    if attempt < max_retry:
                        retry_wait = random.uniform(9.0, 11.0)
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 실패 ({elapsed:.1f}s) "
                              f"/ 재시도 {attempt}/{max_retry} ({retry_wait:.1f}s 후)")
                        time.sleep(retry_wait)
                    else:
                        print(f"  [{i+1}/{len(remaining)}] {game_idx} → 최종 실패 ({max_retry}회 시도)")
                        failed.append(game_idx)

            if records:
                all_records.extend(records)

            # 중간 저장
            if (i + 1) % save_every == 0:
                _save_inning_summary(all_records, output_path)
                print(f"  >>> 중간 저장 완료 ({i+1}/{len(remaining)})")

            # 9~11초 랜덤 딜레이 (응답시간 포함)
            target = random.uniform(9.0, 11.0)
            wait = max(0, target - elapsed)
            time.sleep(wait)

        # 최종 저장
        df = _save_inning_summary(all_records, output_path)
        total_failed[year] = failed

        print(f"\n{year}년 완료: {len(df)}행 저장 → {output_path}")
        print(f"실패 {len(failed)}개: {failed if failed else '없음'}")

    return total_failed


# ==================== 박스스코어 보충 수집 ====================
# 박스스코어(투수/타자/이닝)는 PBP와 달리 최종 실패해도 아무 데도 남지 않았다.
# crawl_*_by_year 는 3회 재시도까지 실패한 game_idx 목록을 돌려주는데,
# run_post_processing 이 그 반환값을 그냥 버렸기 때문이다. 그래서 PBP는 저장됐는데
# 박스스코어만 빠진 경기가 조용히 생길 수 있었고, 나중에 채울 방법도 없었다
# (이미 저장된 경기는 PBP를 다시 크롤링하지 않으므로 run_post_processing 에
#  다시 들어갈 일이 없다).
#
#   check_box_missing()   PBP에는 있는데 박스스코어 파일엔 없는 game_id 목록
#   retry_box_scores()    그 경기들만 다시 긁어 각 박스스코어 파일 끝에 이어붙임
#
# 판정 기준은 로그가 아니라 '실제 파일'이다 — 실패 기록이 남기 시작하기 전에
# 빠진 과거 경기까지 같이 잡힌다. crawl_*_by_year 는 파일에 이미 있는 game_idx를
# 건너뛰므로 몇 번을 돌려도 중복이 생기지 않는다.

BOX_KINDS = {
    # 종류: (폴더 겸 파일 이름, 파일 안의 game_id 열 이름, 수집 함수)
    "pitcher": ("pitcher_box",    "game_idx", crawl_pitcher_box_by_year),
    "batter":  ("batter_box",     "game_idx", crawl_batter_box_by_year),
    "inning":  ("inning_summary", "game_id",  crawl_inning_summary_by_year),
}


def _box_dir(kind):
    return os.path.join(os.path.dirname(BASE_DIR), BOX_KINDS[kind][0])


def _box_path(kind, year):
    return os.path.join(_box_dir(kind), f"{BOX_KINDS[kind][0]}_{year}.xlsx")


def get_saved_box_ids(kind, year):
    """박스스코어 파일에 실제로 들어 있는 game_idx 집합. 파일이 없으면 빈 집합."""
    path = _box_path(kind, year)
    if not os.path.exists(path):
        return set()
    id_col = BOX_KINDS[kind][1]
    try:
        df = pd.read_excel(path, usecols=[id_col])
    except Exception as e:
        print(f"  [경고] {os.path.basename(path)} 읽기 실패, 빈 것으로 간주: {e}")
        return set()
    return set(pd.to_numeric(df[id_col], errors="coerce").dropna().astype(int).tolist())


def get_pbp_game_years():
    """BASE_DIR의 PBP 엑셀을 훑어 {game_id: year} 를 만든다.

    박스스코어 파일은 연도별로 갈리므로(pitcher_box_2026.xlsx) 보충 수집에도
    game_id마다 연도가 필요한데, run_post_processing 은 방금 크롤링한 df에서
    year를 읽어 쓴다. 여기서는 그 df가 없으니 저장된 PBP에서 되읽는다.
    """
    mapping = {}
    if not os.path.exists(BASE_DIR):
        return mapping

    for root, dirs, files in os.walk(BASE_DIR):
        for fname in files:
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue
            try:
                df = pd.read_excel(os.path.join(root, fname), usecols=["game_id", "year"])
            except Exception as e:
                print(f"  [경고] {fname_nfc} 읽기 실패, 스킵: {e}")
                continue
            df = df.dropna(subset=["game_id", "year"])
            for gid, yr in zip(df["game_id"], df["year"]):
                mapping[int(gid)] = int(yr)
    return mapping


def check_box_missing(kinds=None):
    """PBP에는 저장돼 있는데 박스스코어 파일엔 없는 game_id를 종류별로 찾는다.

    반환: {"pitcher": [...], "batter": [...], "inning": [...]}
    """
    kinds = list(kinds) if kinds else list(BOX_KINDS)
    game_years = get_pbp_game_years()
    if not game_years:
        print("PBP 엑셀에서 game_id를 찾지 못했습니다. BASE_DIR을 확인하세요.")
        return {k: [] for k in kinds}

    years = sorted(set(game_years.values()))
    print(f"PBP 저장 경기: {len(game_years)}개 ({', '.join(str(y) for y in years)})\n")

    missing = {}
    for kind in kinds:
        label = BOX_KINDS[kind][0]
        saved = {}
        for y in years:
            saved[y] = get_saved_box_ids(kind, y)
            if not os.path.exists(_box_path(kind, y)):
                print(f"  [{label}] {y}년 파일이 아예 없습니다 — 전부 누락으로 잡힙니다.")
        miss = sorted(gid for gid, yr in game_years.items() if gid not in saved[yr])
        missing[kind] = miss
        print(f"[{label}] 누락 {len(miss)}개: {miss if miss else '없음'}")

    return missing


def _record_box_failures(failed_by_kind):
    """crawl_*_by_year 가 돌려준 {연도: [실패 game_idx]} 를 로그에 모아둔다.

    예전에는 이 반환값을 버려서 최종 실패한 경기가 아무 데도 남지 않았다.
    """
    flat = {kind: sorted({g for lst in per_year.values() for g in lst})
            for kind, per_year in failed_by_kind.items()}
    if not any(flat.values()):
        return

    log = load_log()
    box_failed = log.get("failed_box_ids", {})
    for kind, ids in flat.items():
        if ids:
            box_failed[kind] = sorted(set(box_failed.get(kind, [])) | set(ids))
    log["failed_box_ids"] = box_failed
    save_log(log)

    print("\n⚠ 박스스코어 최종 실패 (log의 failed_box_ids에 기록):")
    for kind, ids in flat.items():
        if ids:
            print(f"   {BOX_KINDS[kind][0]}: {ids}")
    print("   → 나중에 retry_box_scores() 로 보충하세요.")


def retry_box_scores(game_ids=None, kinds=None, max_retry=3):
    """박스스코어만 다시 긁어 이어붙인다. PBP 파일은 건드리지 않는다.

    game_ids=None  → check_box_missing() 이 찾은 누락분을 종류별로 알아서 채운다.
    game_ids=[...] → 그 경기들만, kinds 로 지정한 종류에 대해 재수집한다.
    kinds=None     → 투수/타자/이닝 전부. ("batter",) 처럼 일부만 줄 수도 있다.

    예)
        retry_box_scores()                                  # 누락 전부 보충
        retry_box_scores([38994, 38997])                    # 이 경기만 전부
        retry_box_scores([38994], kinds=("batter",))        # 이 경기의 타자 박스만
    """
    kinds = list(kinds) if kinds else list(BOX_KINDS)
    for kind in kinds:
        if kind not in BOX_KINDS:
            print(f"알 수 없는 종류: {kind} (가능: {', '.join(BOX_KINDS)})")
            return

    game_years = get_pbp_game_years()

    if game_ids is None:
        targets = check_box_missing(kinds)
        print()
    else:
        targets = {kind: [int(g) for g in game_ids] for kind in kinds}

    # PBP에 없는 경기는 연도를 알 수 없어 어느 파일에 넣을지 정할 수 없다.
    unknown = sorted({g for ids in targets.values() for g in ids if g not in game_years})
    if unknown:
        print(f"⚠ PBP 엑셀에 없는 game_id {len(unknown)}개는 연도를 몰라 건너뜁니다: {unknown}")
        print("   PBP부터 채우세요 → retry_failed_games(%s)\n" % unknown)

    still_failed = {}
    for kind in kinds:
        label = BOX_KINDS[kind][0]
        ids = [g for g in targets.get(kind, []) if g in game_years]
        if not ids:
            print(f"\n[{label}] 재수집할 경기 없음")
            still_failed[kind] = []
            continue

        by_year = defaultdict(list)
        for gid in ids:
            by_year[game_years[gid]].append(gid)

        print(f"\n[{label}] {len(ids)}경기 재수집 시작 "
              f"(경기당 약 10초, 예상 {len(ids) * 10 / 60:.1f}분)...")
        failed = BOX_KINDS[kind][2](dict(by_year), output_dir=_box_dir(kind), max_retry=max_retry)
        left = sorted({g for lst in failed.values() for g in lst})
        still_failed[kind] = left
        print(f"[{label}] 완료 — 여전히 실패: {left if left else '없음'}")

    # 이번에 시도한 것은 로그에서 빼고, 여전히 실패한 것만 다시 넣는다.
    log = load_log()
    box_failed = log.get("failed_box_ids", {})
    for kind in kinds:
        tried = set(targets.get(kind, []))
        left = set(still_failed.get(kind, []))
        remain = sorted((set(box_failed.get(kind, [])) - tried) | left)
        if remain:
            box_failed[kind] = remain
        else:
            box_failed.pop(kind, None)
    log["failed_box_ids"] = box_failed
    save_log(log)

    print("\n" + "=" * 60)
    total_left = sum(len(v) for v in still_failed.values())
    print(f"박스스코어 보충 완료 — 여전히 실패: {total_left}개")
    if total_left:
        for kind, ids in still_failed.items():
            if ids:
                print(f"  {BOX_KINDS[kind][0]}: {ids}")
        print("  몇 번을 돌려도 같은 경기가 계속 실패하면, 그 경기 페이지에 기록이")
        print("  아예 없는 것일 수 있습니다. 브라우저로 직접 확인해 보세요:")
        print("    https://www.korea-baseball.com/game/record_detail?game_idx=<번호>")
    print("=" * 60)


# ==================== 빠진 경기 점검 ====================
def check_orphan_game_ids(fix=True):
    """달력에서 발견했는데(log["game_ids"]) 원본 엑셀에도, 실패 목록에도, 일시정지 목록에도
    없는 경기를 찾는다.

    run_pipeline() 일반 모드는 last_collected_date 이후 날짜만 훑고, 지난 경기는
    failed_game_ids 에 있는 것만 다시 시도한다. 그래서 어떤 이유로든 실패 목록에서 빠진
    경기는 아무도 다시 시도하지 않고 조용히 사라진다 — 38328(2026.06.14 제물포고 대
    의왕BC)이 그렇게 넉 달 가까이 빠져 있었다(2026-10-05 발견).

    fix=True 이면
      - 빠진 경기를 failed_game_ids 에 다시 넣는다 → 다음 retry_auto / retry_failed_games 가 잡는다.
        (취소 경기도 여기 섞인다. 취소 경기는 원래도 실패 목록에 남아 있으니 같은 취급이다.)
      - 이미 원본 엑셀에 저장된 경기를 failed_game_ids 에서 뺀다. 재수집에 성공하고도 목록에서
        안 지워진 게 2026-10-05 기준 39개 있었다 — 이게 남아 있으면 같은 경기를 또 붙이게 된다.
    """
    log = load_log()
    saved = get_all_saved_game_ids()
    found = set(log.get("game_ids", []))
    failed = set(log.get("failed_game_ids", []))
    suspended = set(log.get("suspended_game_ids", []))

    orphans = sorted(found - saved - failed - suspended)
    stale_failed = sorted(failed & saved)

    print("\n[빠진 경기 점검]")
    print(f"  발견 {len(found)} / 원본 저장 {len(saved)} / 실패 {len(failed)} / 일시정지 {len(suspended)}")
    print(f"  어디에도 없는 경기: {orphans if orphans else '없음'}")
    print(f"  저장됐는데 실패 목록에 남은 경기: {len(stale_failed)}개")

    if fix and (orphans or stale_failed):
        log["failed_game_ids"] = sorted((failed - set(stale_failed)) | set(orphans))
        save_log(log)
        if orphans:
            print(f"  → {len(orphans)}경기를 실패 목록에 다시 넣었습니다. retry_failed_games({orphans}) 로 재수집하세요.")
        if stale_failed:
            print(f"  → 저장된 {len(stale_failed)}경기를 실패 목록에서 뺐습니다.")
    return orphans, stale_failed

# ==================== run_post_processing: box scores + league backfill ====================
def run_post_processing(success_list):
    if not success_list:
        return

    succeeded_ids_by_year = defaultdict(list)
    for game_id, df in success_list:
        year = int(df["year"].iloc[0])
        succeeded_ids_by_year[year].append(game_id)
    by_year = dict(succeeded_ids_by_year)

    # crawl_*_by_year 의 반환값은 3회 재시도까지 실패한 game_idx 목록이다.
    # 예전에는 이 값을 받지 않아서, PBP는 저장됐는데 박스스코어만 빠진 경기가
    # 아무 흔적 없이 생겼다. 이제 _record_box_failures 로 로그에 남긴다.
    box_failed = {}

    print(f"\n투수 박스스코어 수집 시작 ({len(success_list)}경기)...")
    box_failed["pitcher"] = crawl_pitcher_box_by_year(by_year, output_dir=_box_dir("pitcher"))

    print(f"\n타자 박스스코어 수집 시작 ({len(success_list)}경기)...")
    box_failed["batter"] = crawl_batter_box_by_year(by_year, output_dir=_box_dir("batter"))

    print(f"\n이닝 스코어 수집 시작 ({len(success_list)}경기)...")
    box_failed["inning"] = crawl_inning_summary_by_year(by_year, output_dir=_box_dir("inning"))

    _record_box_failures(box_failed)

    print(f"\nleague(경기 종류) 열 백필 시작 ({len(success_list)}경기)...")
    backfill_league_column(target_game_ids=[game_id for game_id, _ in success_list])


# ==================== 메인 실행 ====================
def run_pipeline(start_date=None, end_date=None):
    """
    start_date / end_date 를 지정하지 않으면 기존과 동일하게
    'last_collected_date + 1 ~ 오늘' 범위로 매일 수집한다 (일반 모드).

    start_date 또는 end_date 중 하나라도 지정하면 '수동 범위 모드'로 동작:
      - 지정한 구간만 수집 (예: run_pipeline(start_date="2026-04-10", end_date="2026-04-20"))
      - 이 구간의 game_idx는 log["game_ids"]에 이미 있어도(=한 번 발견됐던 적이
        있어도) 무조건 다시 후보로 잡는다. 대신 '실제로 엑셀 파일에 저장돼
        있는지'만 확인해서, 진짜로 저장된 경기만 스킵하고 크롤링 실패/일시정지/
        누락 등으로 엑셀에 없는 경기는 전부 재수집 대상에 포함시킨다.
        (get_all_saved_game_ids()로 BASE_DIR의 모든 PBP 엑셀을 직접 스캔해서 판단)
      - PBP/박스스코어는 항상 해당 월의 '실제 마지막 파일'을 읽어 끝에 이어붙이고,
        10경기가 차면 다음 알파벳 파일을 새로 만들어 채운다 (save_games_to_files 로직).
      - last_collected_date는 과거 날짜를 보충 수집해도 거꾸로 당겨지지 않는다.
        (이미 더 최신 날짜까지 수집되어 있다면 그대로 유지, 더 미래 날짜까지
         수집했을 때만 전진(forward) 갱신)
    """
    os.makedirs(BASE_DIR, exist_ok=True)
    log = load_log()

    manual_range = (start_date is not None) or (end_date is not None)

    if manual_range:
        s = _to_date(start_date) if start_date is not None else COLLECTION_START
        e = _to_date(end_date) if end_date is not None else get_kst_today()
        new_ids, collected_end_date = collect_game_ids_for_range(log, s, e)
    else:
        new_ids, collected_end_date = collect_new_game_ids(log)

    if not new_ids:
        print("새로 수집된 경기가 없습니다.")
        check_orphan_game_ids()
        return

    # 2. PBP 크롤링
    print(f"\n총 {len(new_ids)}경기 PBP 크롤링 시작...")
    success_list, fail_list, suspended_list = crawl_games(new_ids)

    if suspended_list:
        existing_suspended = set(log.get("suspended_game_ids", []))
        log["suspended_game_ids"] = list(existing_suspended | set(suspended_list))
        print(f"일시정지 경기 {len(suspended_list)}개 → suspended_game_ids 저장: {suspended_list}")

    if fail_list:
        existing_failed = log.get("failed_game_ids", [])
        log["failed_game_ids"] = list(set(existing_failed + fail_list))

    # 3. 날짜순 정렬 후 PBP 파일 저장 (월별 마지막 파일에 append, 10개 차면 다음 알파벳)
    if success_list:
        success_list.sort(key=lambda x: x[1]["date"].iloc[0])
        save_games_to_files(success_list, log)

    # 4. last_collected_date 갱신
    #    - 일반 모드: 항상 collected_end_date로 갱신 (기존 동작과 동일)
    #    - 수동 범위 모드: 과거 구간을 보충 수집해도 last_collected_date가
    #      거꾸로 당겨지지 않도록, '더 전진'하는 경우에만 갱신
    if collected_end_date:
        prev_last = log.get("last_collected_date")
        if not manual_range:
            log["last_collected_date"] = collected_end_date.isoformat()
        else:
            if prev_last is None or collected_end_date > date.fromisoformat(prev_last):
                log["last_collected_date"] = collected_end_date.isoformat()
            else:
                print(f"(수동 범위 모드: last_collected_date는 기존 값({prev_last}) 유지)")
    save_log(log)

    # 5. box score (pitcher/batter) + inning summary + league backfill -> run_post_processing
    run_post_processing(success_list)

    print("\n" + "=" * 60)
    print("파이프라인 완료!")
    print(f"성공: {len(success_list)}경기 / 일시정지: {len(suspended_list)}경기 / 실패: {len(fail_list)}경기")
    if suspended_list:
        print(f"일시정지 목록 (suspended_game_ids에 저장됨): {suspended_list}")
    if fail_list:
        print(f"실패 목록 (로그에 저장됨): {fail_list}")
    print("=" * 60)

    # 발견만 되고 어디에도 안 들어간 경기가 없는지 매번 확인한다
    check_orphan_game_ids()


# ==================== 실행 ====================
# 0) 기존 파일들에 예외_아웃 / RBI / league 열 일괄 추가 (한 번만 실행하면 됨, 이미 있는 파일은 스킵)
#    league는 run_pipeline() 내에서 신규 수집 경기에 한해 자동으로도 채워지므로,
#    과거에 이미 저장된 경기들을 한 번에 채우고 싶을 때만 아래를 수동 실행하면 된다.
# backfill_예외_아웃_column()
# backfill_RBI_column()
# backfill_league_column()

# 1) 일반 실행 (매일, 기존과 동일)
# run_pipeline()

# 2) 과거 특정 날짜 구간을 다시 수집하고 싶을 때 (예: 4월 10일~20일)
#    이 구간의 game_idx는 한 번 발견된 적이 있어도 무조건 다시 후보로 잡고,
#    실제 엑셀 파일에 저장돼 있는지를 직접 스캔해서 확인한다.
#    이미 저장된 경기만 스킵되고, 실패/일시정지/누락 등으로 엑셀에 없는 경기는
#    전부 재수집되어 해당 월의 마지막 엑셀 파일 끝에 이어붙고,
#    10경기가 차면 다음 알파벳 파일이 새로 생성된다.
# run_pipeline(start_date="2026-07-06", end_date="2026-07-06")

# 3) 특정 시작일부터 오늘까지만 다시 긁고 싶을 때 (end_date 생략 시 오늘까지)
# run_pipeline(start_date="2026-05-01")

# 일시정지 경기 재시도 (경기 재개 후 실행)
# retry_suspended_games()
# retry_suspended_games(
#     target_file_path="/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/2026/6월/2026_06_a.xlsx"
# )

# 빠진 경기 점검 (run_pipeline() 끝에서 자동으로 돈다. 따로 확인만 할 때는 fix=False)
# check_orphan_game_ids(fix=False)

# 수동 재처리 (월별 마지막 파일에 자동 append)
# retry_failed_games([39231, 39232])   # 처리 끝남 — 다시 돌리면 원본에 중복으로 붙는다

# 38328 재수집 (2026.06.14 제물포고 9:1 의왕BC, 콜드).
# 경기 번호는 수집됐는데 원본 파일에도, 실패 목록에도 없어서 재시도 대상에서 빠졌던 경기.
# retry_failed_games 는 원본과의 중복 검사를 하지 않으므로, 이미 저장됐으면 건너뛴다.
TARGET = [38328]
already = sorted(set(TARGET) & get_all_saved_game_ids())
if already:
    print(f"이미 원본 파일에 있습니다: {already} — 재수집하지 않습니다.")
else:
    retry_failed_games(TARGET)
    saved = get_all_saved_game_ids()
    print("\n[확인] 원본 파일 저장:", {g: (g in saved) for g in TARGET})
    missing = check_box_missing(list(BOX_KINDS))
    left = {k: sorted(set(v) & set(TARGET)) for k, v in missing.items() if set(v) & set(TARGET)}
    print("[확인] 박스스코어 누락:", left if left else "없음")
    if left:
        print("  → retry_box_scores(TARGET) 로 보충하세요.")

# ── 박스스코어(투수/타자/이닝)만 보충 ──────────────────────────────
# PBP는 저장됐는데 박스스코어 수집이 최종 실패한 경기를 나중에 채울 때 쓴다.
# 실제 파일을 스캔해서 판단하므로, 먼저 무엇이 빠졌는지 확인만 해봐도 된다.
# check_box_missing()
#
# 누락된 것 전부 보충 (경기당 약 10초)
# retry_box_scores()
#
# 특정 경기만
# retry_box_scores([38994, 38997])
#
# 특정 경기의 타자 박스스코어만
# retry_box_scores([39125], kinds=("batter","pitcher"))

# 수동 재처리 (특정 파일 지정)
# retry_failed_games(
#     [38329],
#     target_file_path="/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/2026/6월/2026_06_i.xlsx"
# )

# 자동으로 작업자 파일 - 파일 체크 시트 파일 추가

In [ ]:
import os
import re
import unicodedata
import gspread
from google.colab import auth
from googleapiclient.discovery import build
from google.auth import default

# ==================== 인증 ====================
auth.authenticate_user()
creds, _ = default()

# ==================== 설정 ====================
ROOT_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
GSHEET_NAME = "작업자 현황 및 파일 체크"
SHEET_NAME = "파일 체크"

YEAR_FOLDER_PATTERN = re.compile(r'^\d{4}$')       # 2020, 2022, 2023 ... 형식
MONTH_FOLDER_PATTERN = re.compile(r'^\d{1,2}월$')  # 3월, 4월 ... 형식


# ==================== 전체 연도의 {year}/{month}월/ 폴더 안 파일명 수집 ====================
def get_all_filenames(root_dir):
    filenames = []
    root_dir_nfc = unicodedata.normalize('NFC', root_dir)

    if not os.path.isdir(root_dir_nfc):
        print(f"  경고: 폴더가 존재하지 않습니다 - {root_dir_nfc}")
        return filenames

    # 루트 아래 연도 폴더만 확인
    for year_entry in sorted(os.listdir(root_dir_nfc)):
        year_entry_nfc = unicodedata.normalize('NFC', year_entry)
        year_path = os.path.join(root_dir_nfc, year_entry_nfc)

        if not os.path.isdir(year_path) or not YEAR_FOLDER_PATTERN.match(year_entry_nfc):
            continue

        # 연도 폴더 아래 "N월" 폴더만 확인
        for month_entry in sorted(os.listdir(year_path)):
            month_entry_nfc = unicodedata.normalize('NFC', month_entry)
            month_path = os.path.join(year_path, month_entry_nfc)

            if not os.path.isdir(month_path) or not MONTH_FOLDER_PATTERN.match(month_entry_nfc):
                continue

            # 월 폴더 안의 xlsx만 수집 (하위 폴더까지 포함해서 안전하게)
            for root, dirs, files in os.walk(month_path):
                dirs[:] = sorted([unicodedata.normalize('NFC', d) for d in dirs])
                for file in sorted(files):
                    file_nfc = unicodedata.normalize('NFC', file)
                    if file_nfc.endswith('.xlsx') and not file_nfc.startswith('~$'):
                        filename_no_ext = os.path.splitext(file_nfc)[0]
                        filenames.append(filename_no_ext)

    filenames.sort()
    return filenames


# ==================== gsheet에서 파일 찾기 ====================
def find_gsheet_by_name(sheet_name, creds):
    service = build('drive', 'v3', credentials=creds)
    query = f"name='{sheet_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    if not files:
        raise FileNotFoundError(f"'{sheet_name}' 파일을 Drive에서 찾을 수 없습니다.")
    if len(files) > 1:
        print(f"  동일 이름 파일 {len(files)}개 발견, 첫 번째 파일 사용: {files[0]['id']}")
    return files[0]['id']


# ==================== 기존 파일명 목록 가져오기 ====================
def get_existing_filenames(worksheet):
    """파일체크 시트 A열에서 이미 있는 파일명 목록 반환"""
    all_values = worksheet.col_values(1)  # A열 전체
    existing = [v.strip() for v in all_values[1:] if v.strip()]
    return existing


# ==================== 메인 실행 ====================
def update_file_check_sheet():
    # 1. 전체 연도의 'N월' 하위 폴더에서 파일명 수집
    print("전체 연도 폴더의 'N월' 하위 폴더에서 파일명 수집 중...")
    filenames = get_all_filenames(ROOT_DIR)
    print(f"  수집된 파일 수: {len(filenames)}개")
    if filenames:
        print(f"  범위: {filenames[0]} ~ {filenames[-1]}")

    # 2. gsheet 연결
    print(f"\n'{GSHEET_NAME}' gsheet 검색 중...")
    gc = gspread.authorize(creds)
    sheet_id = find_gsheet_by_name(GSHEET_NAME, creds)
    spreadsheet = gc.open_by_key(sheet_id)
    worksheet = spreadsheet.worksheet(SHEET_NAME)
    print(f"  연결 완료 (ID: {sheet_id})")

    # 3. 기존 파일명 확인 (중복 방지)
    existing_filenames = get_existing_filenames(worksheet)
    print(f"\n기존 파일체크 시트 항목 수: {len(existing_filenames)}개")

    # 4. 새로 추가할 파일명 필터링
    new_filenames = [f for f in filenames if f not in existing_filenames]
    print(f"새로 추가할 파일 수: {len(new_filenames)}개")

    if not new_filenames:
        print("추가할 새 파일이 없습니다.")
        return

    # 5. 시작 행 계산
    next_row = len(existing_filenames) + 2  # 헤더 1행 + 기존 데이터

    # 6. 행 수 부족하면 자동 확장
    current_rows = worksheet.row_count
    needed_rows = next_row + len(new_filenames) - 1
    if needed_rows > current_rows:
        worksheet.add_rows(needed_rows - current_rows)
        print(f"  시트 행 확장: {current_rows} → {needed_rows}")

    # 7. 업데이트
    print(f"{next_row}행부터 추가 시작...")
    cell_range = f"A{next_row}:A{next_row + len(new_filenames) - 1}"
    values = [[f] for f in new_filenames]
    worksheet.update(values=values, range_name=cell_range)

    print(f"\n완료! {len(new_filenames)}개 파일명 추가됨")
    print(f"  추가 범위: {new_filenames[0]} ~ {new_filenames[-1]}")


update_file_check_sheet()

In [ ]:
# # ==================== 수비 교체 타이밍 수기입력 템플릿 생성 ====================
# # batter_box에서 각 타순 슬롯의 '첫 선수 / 첫 포지션'은 선발이라 타이밍이 필요 없다.
# # 그 뒤에 나오는 것만 변경 이벤트(교체투입 / 포지션변경)이고, 이벤트마다 한 행을 만든다.
# # 구조(경기/팀/타순/순서)만 채워두고, 기록지에서 옮겨 적을 칸은 비워둔다.
# #
# #   기록지 표기 예:  "2 윤형준 79"  (윤형준의 등번호가 44번이라면)
# #     = 수비번호 2(포수) 윤형준이 7회 9번타자 타석 '결과 전'에 투입
# #     -> player_no=44, pos_no=2, player_name=윤형준, inning=7, at_bat_order=9
# #     입력 칸 순서: 등번호 - 수비번호 - 이름 - 이닝 - 상대타순
# #     등번호를 맨 앞에 둔 것은 동명이인 때문에 이름만으로는 선수를 특정할 수 없어서다.
# #     기록지에 번호가 안 적혀 있으면 비워둔다(나중에 '번호없음'은 100으로 채운다).
# #     cycle = '타자일순이 몇 번 돌았는지'(0부터). 몇 바퀴째가 아니라 일순 '횟수'다.
# #       기록지 '4.12' = 4회 / 일순 1회 / 2번타자 -> inning=4, at_bat_order=2, cycle=1
# #         => 그 이닝 2번타자의 '두 번째' 타석을 가리킨다.
# #       일순이 안 돌았으면 0 (비워도 0으로 처리). 계산: 그 이닝에서 해당 타순의
# #       타석들 중 (cycle+1)번째를 고른다.  (일순 도는 이닝은 전체의 2.3% 뿐)
# #   갑지 = 초공격지, 을지 = 말공격지. 수비팀은 그 반대편이므로 topbot 은 코드가 자동 처리.
# #   수비번호: 1투수 2포수 3일루 4이루 5삼루 6유격 7좌익 8중견 9우익 (position 은 이미 채워둠)
# #
# #   ※ 파일은 연도 폴더(<BASE_DIR>/<YEAR>/) 에 둔다. 예전에는 players_by_season/ 에 있었고,
# #     거기 파일이 남아 있으면 한 번 읽어서 새 위치로 옮겨 준다(옛 파일은 지우지 않는다).
# #   ※ 파일이 이미 있으면 덮어쓰지 않고 병합한다 — 입력해둔 값은 그대로 두고,
# #     새로 생긴 경기의 이벤트 행만 추가한다.
# #   ※ 옛 형식은 player_name 에 '최은택(13)' 처럼 이름과 등번호가 붙어 있었다.
# #     읽을 때 자동으로 player_no / player_name 으로 갈라준다.
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

# import os
# import re
# import unicodedata
# import pandas as pd


# # ==================== 설정 ====================
# BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
# DATA_DIR = unicodedata.normalize("NFC", os.path.join(BASE_DIR, "players_by_season"))
# YEAR = 2026

# # batter_box / season_full 후보 경로 (실제 위치가 다르면 여기에 추가)
# BATTER_BOX_CANDIDATES = [
#     f"{BASE_DIR}/batter_box/batter_box_{YEAR}.xlsx",   # <- 실제 위치
#     f"{BASE_DIR}/batter_box_{YEAR}.xlsx",
#     f"{BASE_DIR}/players_by_season/batter_box_{YEAR}.xlsx",
#     f"{BASE_DIR}/season_full/batter_box_{YEAR}.xlsx",
#     f"{BASE_DIR}/{YEAR}/batter_box_{YEAR}.xlsx",
# ]
# SCHEDULE_CANDIDATES = [
#     f"{BASE_DIR}/season_full/schedule_{YEAR}.xlsx",   # game_id -> date 는 일정표에서만 가져온다
#     f"{BASE_DIR}/schedule/schedule_{YEAR}.xlsx",
#     f"{BASE_DIR}/schedule_{YEAR}.xlsx",
# ]
# # 연도 폴더 안에 둔다. 예전 위치(players_by_season)는 이어받기용으로만 본다.
# OUT_FILE = unicodedata.normalize("NFC", os.path.join(BASE_DIR, str(YEAR), f"defense_sub_manual_{YEAR}.xlsx"))
# LEGACY_OUT_FILES = [
#     unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"defense_sub_manual_{YEAR}.xlsx")),
# ]

# COLS = ["game_id", "date", "def_team",
#         "player_no", "pos_no", "player_name", "inning", "at_bat_order", "cycle", "memo"]
# INPUT_COLS = ["player_no", "pos_no", "player_name", "inning", "at_bat_order", "cycle"]  # 손으로 채우는 칸
# KEY = ["game_id", "def_team"]        # 병합은 경기·팀 단위(행 개수)로만 맞춘다

# # 수비번호 -> 포지션 (기록지 표기 기준). 나중에 계산 단계에서 이 표로 변환한다.
# # ※ 1(투수)은 투구이닝이라 '수비이닝' 집계 대상이 아니다 — 변환은 하되 집계에선 제외.
# POS_NO_MAP = {1: "투수", 2: "포수", 3: "1루수", 4: "2루수", 5: "3루수",
#               6: "유격수", 7: "좌익수", 8: "중견수", 9: "우익수"}

# # '최은택(13)' 형태. 기록지 오타로 닫는 괄호가 빠진 '이재범(18' 도 18 로 읽는다.
# _NAME_NO_RE = re.compile(r"^\s*(.*?)\s*\(\s*(\d+)\s*\)?\s*$")
# _EMPTYISH = {"", "nan", "NaN", "<NA>", "None"}


# def _resolve(path):
#     """Drive(NFC/NFD 혼재) 관대 탐색. 없으면 None."""
#     path = unicodedata.normalize("NFC", path)
#     if os.path.exists(path):
#         return path
#     d, base = os.path.dirname(path), os.path.basename(path)
#     if not os.path.isdir(d):
#         return None
#     want = unicodedata.normalize("NFC", base)
#     for f in os.listdir(d):
#         if unicodedata.normalize("NFC", f) == want:
#             return os.path.join(d, f)
#     return None


# def _first(cands):
#     for c in cands:
#         p = _resolve(c)
#         if p:
#             return p
#     return None


# def _split_name_no(v):
#     """'최은택(13)' -> (13, '최은택'). 번호 표기가 없으면 (None, 원문)."""
#     if pd.isna(v):
#         return (None, v)
#     s = str(v).strip()
#     if not s:
#         return (None, "")
#     m = _NAME_NO_RE.match(s)
#     if m:
#         return (int(m.group(2)), m.group(1))
#     return (None, s)


# def _to_new_schema(df):
#     """옛 파일을 현재 스키마로 맞춘다 — team->def_team, '이름(등번호)' 분리, 컬럼 순서."""
#     df = df.copy()
#     if "team" in df.columns and "def_team" not in df.columns:
#         df = df.rename(columns={"team": "def_team"})   # 더 예전 파일 호환
#     for c in COLS:
#         if c not in df.columns:
#             df[c] = ""

#     split_n = 0
#     no_col, name_col = [], []
#     for no, name in zip(df["player_no"], df["player_name"]):
#         if str(no).strip() not in _EMPTYISH:      # 이미 갈라진 행은 손대지 않는다
#             no_col.append(no)
#             name_col.append(name)
#             continue
#         parsed_no, parsed_name = _split_name_no(name)
#         if parsed_no is None:
#             no_col.append(no)
#             name_col.append(name)
#         else:
#             no_col.append(parsed_no)
#             name_col.append(parsed_name)
#             split_n += 1
#     df["player_no"], df["player_name"] = no_col, name_col
#     if split_n:
#         print(f"옛 형식 '이름(등번호)' {split_n}행을 player_no / player_name 으로 분리")
#     return df[COLS]


# def build_sub_template():
#     bpath = _first(BATTER_BOX_CANDIDATES)
#     if not bpath:
#         print("batter_box 를 찾지 못했습니다. 아래 후보를 확인하거나 경로를 추가하세요:")
#         for c in BATTER_BOX_CANDIDATES:
#             print("   -", c)
#         return
#     print("batter_box:", bpath)
#     bb = pd.read_excel(bpath, engine="openpyxl").rename(
#         columns={"game_idx": "game_id", "batting_order": "order", "player_name": "name"})

#     # 경기 날짜(표시용). inning 은 자동으로 채우지 않는다 — 전부 기록지에서 직접 입력.
#     # season_full 대신 일정표(schedule)에서 game_id->date 만 가져온다(가볍고 목적에 맞음).
#     date_of = {}
#     spath = _first(SCHEDULE_CANDIDATES)
#     if spath:
#         sch = pd.read_excel(spath, engine="openpyxl")
#         if {"game_id", "date"}.issubset(sch.columns):
#             date_of = sch.drop_duplicates("game_id").set_index("game_id")["date"].to_dict()
#         print("schedule:", spath)

#     FIELD_SKIP = {"투수", "지명타자", "지명 타자"}   # 수비이닝 대상 아님(투수=pitcher_box, DH=수비 안 함)
#     rows = []
#     for (g, t), gsub in bb.groupby(["game_id", "team"], sort=False):
#         for o, grp in gsub.groupby("order", sort=True):
#             grp = grp.reset_index(drop=True)
#             for i, r in grp.iterrows():
#                 chain = [x.strip() for x in str(r["position"]).split("/") if x.strip()]
#                 taken = []
#                 if i > 0 and chain:
#                     taken.append(chain[0])        # 교체 투입 시 맡은 포지션
#                 taken.extend(chain[1:])           # 이후 포지션 변경들
#                 for pos in taken:
#                     if pos in FIELD_SKIP:         # 투수·지명타자 제외
#                         continue
#                     rows.append({
#                         "game_id": g, "date": date_of.get(g, ""), "def_team": t,
#                         "player_no": "", "pos_no": "", "player_name": "",
#                         "inning": "", "at_bat_order": "", "cycle": "", "memo": "",
#                     })

#     new = pd.DataFrame(rows)
#     if new.empty:
#         print("변경 이벤트가 없습니다.")
#         return

#     # 기존 입력 보존: 이미 입력한 (경기,팀)은 그 블록을 통째로 유지하고,
#     # 새로 생긴 (경기,팀)만 빈 행으로 추가한다.
#     cur = _resolve(OUT_FILE)
#     opath = cur or _first(LEGACY_OUT_FILES)
#     if opath:
#         if cur is None:
#             print(f"옛 위치에서 이어받습니다: {opath}")
#             print(f"  -> 앞으로는 {OUT_FILE} 에 저장합니다 (옛 파일은 지우지 않으니 확인 후 정리하세요)")
#         old = _to_new_schema(pd.read_excel(opath, engine="openpyxl"))
#         filled_keys = set(map(tuple, old.loc[
#             old[INPUT_COLS].astype(str).apply(lambda s: s.str.strip().ne("").any(), axis=1),
#             ["game_id", "def_team"]].values))
#         if filled_keys:
#             keep_old = old[old[["game_id", "def_team"]].apply(tuple, axis=1).isin(filled_keys)]
#             new = new[~new[["game_id", "def_team"]].apply(tuple, axis=1).isin(filled_keys)]
#             new = pd.concat([keep_old, new], ignore_index=True)
#             print(f"기존 입력된 {len(filled_keys)}개 (경기,팀) 블록 보존")

#     new = new.sort_values(["date", "game_id", "def_team"])
#     out = new[COLS]
#     os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)
#     out.to_excel(OUT_FILE, index=False, engine="openpyxl")

#     filled = (out["inning"].astype(str).str.strip() != "").sum()
#     print("\n" + "=" * 60)
#     print(f"템플릿 생성 완료: {OUT_FILE}")
#     print(f"  변경 이벤트 {len(out)}행 / 이닝 입력된 행 {filled}")
#     print("  입력할 칸: player_no - pos_no - player_name - inning - at_bat_order")
#     print("             ('2 윤형준 79', 윤형준 44번 -> 44 / 2 / 윤형준 / 7 / 9)")
#     print("=" * 60)
#     print(out.head(10).to_string(index=False))


# # ==================== 실행 ====================
# build_sub_template()
